In [11]:
import os
os.environ["OPENAI_API_KEY"] = "XX"

In [3]:
# Run once if needed:
!pip install -U openai pandas tqdm python-dotenv

# Optional later:
!pip install -U anthropic google-genai

     |████████████████████████████████| 1.2 MB 4.2 MB/s            
     |████████████████████████████████| 10.8 MB 37.6 MB/s            
     |████████████████████████████████| 78 kB 19.9 MB/s            
     |████████████████████████████████| 315 kB 163.2 MB/s            
     |████████████████████████████████| 44 kB 14.7 MB/s            
  Attempting uninstall: typing-extensions
    Found existing installation: typing-extensions 4.13.2
    Uninstalling typing-extensions-4.13.2:
      Successfully uninstalled typing-extensions-4.13.2
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.1
    Uninstalling tqdm-4.67.1:
      Successfully uninstalled tqdm-4.67.1
  Attempting uninstall: jiter
    Found existing installation: jiter 0.9.0
    Uninstalling jiter-0.9.0:
      Successfully uninstalled jiter-0.9.0
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.1.0
    Uninstalling python-dotenv-1.1.0:
      Successfully uninstalled py

     |████████████████████████████████| 7.9 MB 33.5 MB/s            
     |████████████████████████████████| 180 kB 35.4 MB/s            
  Attempting uninstall: cffi
    Found existing installation: cffi 1.16.0
    Uninstalling cffi-1.16.0:
      Successfully uninstalled cffi-1.16.0
  Attempting uninstall: cryptography
    Found existing installation: cryptography 3.4.8
    Uninstalling cryptography-3.4.8:
      Successfully uninstalled cryptography-3.4.8
  Attempting uninstall: websockets
    Found existing installation: websockets 11.0.3
    Uninstalling websockets-11.0.3:
      Successfully uninstalled websockets-11.0.3
  Attempting uninstall: google-auth
    Found existing installation: google-auth 1.35.0
    Uninstalling google-auth-1.35.0:
      Successfully uninstalled google-auth-1.35.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorboard 2.6.0 

In [12]:
from __future__ import annotations

import os
import re
import json
import time
import hashlib
import itertools
import datetime as dt
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any, List, Iterable

import pandas as pd
from tqdm.auto import tqdm

In [13]:
PILOT_DIR = Path("human_data/pilot")
AI_DATA_DIR = Path("ai_data/story_generations")
AI_DATA_DIR.mkdir(parents=True, exist_ok=True)

selected_prompts_path = PILOT_DIR / "writing_prompts_selected_prompts.csv"

selected_story_prompts = pd.read_csv(selected_prompts_path)

selected_story_prompts

,prompt_id,prompt,n_human_stories,median_human_story_words,mean_human_story_words,median_human_story_sentences,mean_human_story_sentences
0,10491,A short Horror story . Something to chill the ...,35,104.0,115.60000,10.0,9.942857
1,93742,[ FF ] 100 Words or Less - The parachute is n'...,32,107.0,119.96875,14.0,13.968750
2,93855,[ FF ] Describe 100 years of a character 's li...,20,112.0,115.60000,10.5,11.700000


In [14]:
SELECTED_PROMPT_IDS = [10491, 93742, 93855]

selected_story_prompts = (
    selected_story_prompts
    .query("prompt_id in @SELECTED_PROMPT_IDS")
    .sort_values("prompt_id")
    .reset_index(drop=True)
)

selected_story_prompts[["prompt_id", "prompt", "n_human_stories"]]

,prompt_id,prompt,n_human_stories
0,10491,A short Horror story . Something to chill the ...,35
1,93742,[ FF ] 100 Words or Less - The parachute is n'...,32
2,93855,[ FF ] Describe 100 years of a character 's li...,20


In [15]:
def generate_all_personas():
    trait_pairs = [
        ("extroverted", "introverted"),
        ("agreeable", "antagonistic"),
        ("conscientious", "unconscientious"),
        ("neurotic", "emotionally_stable"),
        ("open to experience", "closed to experience"),
    ]
    return list(itertools.product(*trait_pairs))


def persona_tuple_to_id(persona_tuple) -> str:
    return "__".join(
        t.replace(" ", "_").replace("-", "_")
        for t in persona_tuple
    )


def persona_tuple_to_prompt_text(persona_tuple) -> str:
    traits = ", ".join(persona_tuple)
    return (
        "Write as if you are a person with the following personality profile: "
        f"{traits}. Let this personality influence the creative choices, tone, "
        "voice, imagery, pacing, and emotional framing of the story, while still "
        "following the user's writing prompt exactly."
    )


ALL_PERSONAS = generate_all_personas()

len(ALL_PERSONAS), ALL_PERSONAS[:3]

(32,
 [('extroverted',
   'agreeable',
   'conscientious',
   'neurotic',
   'open to experience'),
  ('extroverted',
   'agreeable',
   'conscientious',
   'neurotic',
   'closed to experience'),
  ('extroverted',
   'agreeable',
   'conscientious',
   'emotionally_stable',
   'open to experience')])

In [16]:
NEUTRAL_SYSTEM_INSTRUCTIONS = (
    "You are participating in a creative writing task. "
    "Respond to the prompt as a human participant would. "
    "Follow the prompt's constraints exactly. "
    "Do not explain your answer. "
    "Do not include commentary before or after the creative response."
)


def build_system_instructions(
    condition_type: str,
    persona_tuple: Optional[tuple] = None,
) -> str:
    """
    condition_type:
        - 'neutral'
        - 'personality'
    """
    if condition_type == "neutral":
        return NEUTRAL_SYSTEM_INSTRUCTIONS

    if condition_type == "personality":
        if persona_tuple is None:
            raise ValueError("persona_tuple must be provided for personality condition.")

        return (
            NEUTRAL_SYSTEM_INSTRUCTIONS
            + "\n\n"
            + persona_tuple_to_prompt_text(persona_tuple)
        )

    raise ValueError(f"Unknown condition_type: {condition_type}")


def build_user_prompt(story_prompt: str) -> str:
    return f"Prompt:\n{story_prompt}"

In [17]:
def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def stable_hash(obj: Any) -> str:
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def successful_request_keys(path: Path) -> set:
    records = read_jsonl(path)
    return {
        r["request_key"]
        for r in records
        if r.get("status") == "success" and r.get("text")
    }


def make_request_key(
    provider: str,
    model: str,
    scenario_name: str,
    prompt_id: int,
    condition_type: str,
    temperature: float,
    run_idx: int,
    persona_id: Optional[str],
    system_instructions: str,
    user_prompt: str,
) -> str:
    return stable_hash(
        {
            "provider": provider,
            "model": model,
            "scenario_name": scenario_name,
            "prompt_id": int(prompt_id),
            "condition_type": condition_type,
            "temperature": float(temperature),
            "run_idx": int(run_idx),
            "persona_id": persona_id,
            "system_instructions": system_instructions,
            "user_prompt": user_prompt,
        }
    )

In [18]:
@dataclass
class GenerationResult:
    text: str
    raw_response: Optional[Dict[str, Any]] = None
    usage: Optional[Dict[str, Any]] = None
    provider_response_id: Optional[str] = None


class BaseLLMProvider:
    provider_name: str = "base"

    def generate(
        self,
        model: str,
        system_instructions: str,
        user_prompt: str,
        temperature: float,
        max_output_tokens: int,
    ) -> GenerationResult:
        raise NotImplementedError


class OpenAIProvider(BaseLLMProvider):
    provider_name = "openai"

    def __init__(self, api_key_env: str = "OPENAI_API_KEY"):
        from openai import OpenAI
        api_key = os.getenv(api_key_env)
        if not api_key:
            raise EnvironmentError(
                f"Missing {api_key_env}. Set it before running generation."
            )
        self.client = OpenAI(api_key=api_key)

    def generate(
        self,
        model: str,
        system_instructions: str,
        user_prompt: str,
        temperature: float,
        max_output_tokens: int = 800,
    ) -> GenerationResult:
        response = self.client.responses.create(
            model=model,
            instructions=system_instructions,
            input=user_prompt,
            temperature=temperature,
            max_output_tokens=max_output_tokens,
        )

        text = getattr(response, "output_text", None)
        if text is None:
            # Fallback for SDK shape changes.
            text = str(response)

        usage = None
        if hasattr(response, "usage") and response.usage is not None:
            try:
                usage = response.usage.model_dump()
            except Exception:
                usage = dict(response.usage)

        try:
            raw_response = response.model_dump()
        except Exception:
            raw_response = None

        return GenerationResult(
            text=text.strip(),
            raw_response=raw_response,
            usage=usage,
            provider_response_id=getattr(response, "id", None),
        )


class AnthropicProvider(BaseLLMProvider):
    provider_name = "anthropic"

    def __init__(self, api_key_env: str = "ANTHROPIC_API_KEY"):
        raise NotImplementedError("Add Anthropic implementation later.")


class GeminiProvider(BaseLLMProvider):
    provider_name = "gemini"

    def __init__(self, api_key_env: str = "GEMINI_API_KEY"):
        raise NotImplementedError("Add Gemini implementation later.")

In [19]:
# Make sure OPENAI_API_KEY is set in your shell or .env environment before running.
# Example in terminal:
# export OPENAI_API_KEY="..."

provider = OpenAIProvider()

MODEL_NAME = "gpt-5.4"  # change to your target model
PROVIDER_NAME = provider.provider_name

PROVIDER_NAME, MODEL_NAME

('openai', 'gpt-5.4')

In [20]:
@dataclass
class GenerationScenario:
    scenario_name: str
    condition_type: str              # neutral or personality
    temperatures: List[float]
    n_per_prompt: int
    prompt_ids: List[int]
    include_personas: bool = False
    max_output_tokens: int = 800
    description: str = ""


SCENARIOS = {
    # Main benchmark: neutral, T=1, 50 generations per prompt.
    "neutral_main_t1": GenerationScenario(
        scenario_name="neutral_main_t1",
        condition_type="neutral",
        temperatures=[1.0],
        n_per_prompt=50,
        prompt_ids=SELECTED_PROMPT_IDS,
        include_personas=False,
        max_output_tokens=800,
        description="Primary benchmark: neutral prompt, temperature 1.0, 50 generations per prompt.",
    ),

    # Temperature robustness: neutral T=0.7 and T=1.3.
    # We do not need to regenerate T=1.0 here because neutral_main_t1 already exists.
    "neutral_temperature_robustness": GenerationScenario(
        scenario_name="neutral_temperature_robustness",
        condition_type="neutral",
        temperatures=[0.7, 1.3],
        n_per_prompt=10,
        prompt_ids=SELECTED_PROMPT_IDS,
        include_personas=False,
        max_output_tokens=800,
        description="Temperature robustness: neutral prompt at T=0.7 and T=1.3.",
    ),

    # Personality robustness: 32 personas × 3 temperatures × 3 prompts × 10 generations.
    "personality_grid": GenerationScenario(
        scenario_name="personality_grid",
        condition_type="personality",
        temperatures=[0.7, 1.0, 1.3],
        n_per_prompt=10,
        prompt_ids=SELECTED_PROMPT_IDS,
        include_personas=True,
        max_output_tokens=800,
        description="Personality robustness: 32 Big-Five-style profiles crossed with temperature.",
    ),
}

SCENARIOS

{'neutral_main_t1': GenerationScenario(scenario_name='neutral_main_t1', condition_type='neutral', temperatures=[1.0], n_per_prompt=50, prompt_ids=[10491, 93742, 93855], include_personas=False, max_output_tokens=800, description='Primary benchmark: neutral prompt, temperature 1.0, 50 generations per prompt.'),
 'neutral_temperature_robustness': GenerationScenario(scenario_name='neutral_temperature_robustness', condition_type='neutral', temperatures=[0.7, 1.3], n_per_prompt=10, prompt_ids=[10491, 93742, 93855], include_personas=False, max_output_tokens=800, description='Temperature robustness: neutral prompt at T=0.7 and T=1.3.'),
 'personality_grid': GenerationScenario(scenario_name='personality_grid', condition_type='personality', temperatures=[0.7, 1.0, 1.3], n_per_prompt=10, prompt_ids=[10491, 93742, 93855], include_personas=True, max_output_tokens=800, description='Personality robustness: 32 Big-Five-style profiles crossed with temperature.')}

In [21]:
def build_generation_plan(
    scenario: GenerationScenario,
    prompts_df: pd.DataFrame,
    provider_name: str,
    model_name: str,
) -> pd.DataFrame:
    rows = []

    prompts_use = prompts_df[prompts_df["prompt_id"].isin(scenario.prompt_ids)].copy()

    if scenario.include_personas:
        persona_tuples = ALL_PERSONAS
    else:
        persona_tuples = [None]

    for _, prompt_row in prompts_use.iterrows():
        prompt_id = int(prompt_row["prompt_id"])
        story_prompt = prompt_row["prompt"]
        user_prompt = build_user_prompt(story_prompt)

        for temperature in scenario.temperatures:
            for persona_tuple in persona_tuples:
                if persona_tuple is None:
                    persona_id = None
                    persona_traits = None
                else:
                    persona_id = persona_tuple_to_id(persona_tuple)
                    persona_traits = list(persona_tuple)

                system_instructions = build_system_instructions(
                    condition_type=scenario.condition_type,
                    persona_tuple=persona_tuple,
                )

                for run_idx in range(scenario.n_per_prompt):
                    request_key = make_request_key(
                        provider=provider_name,
                        model=model_name,
                        scenario_name=scenario.scenario_name,
                        prompt_id=prompt_id,
                        condition_type=scenario.condition_type,
                        temperature=temperature,
                        run_idx=run_idx,
                        persona_id=persona_id,
                        system_instructions=system_instructions,
                        user_prompt=user_prompt,
                    )

                    rows.append(
                        {
                            "request_key": request_key,
                            "scenario_name": scenario.scenario_name,
                            "provider": provider_name,
                            "model": model_name,
                            "condition_type": scenario.condition_type,
                            "prompt_id": prompt_id,
                            "story_prompt": story_prompt,
                            "temperature": float(temperature),
                            "run_idx": int(run_idx),
                            "persona_id": persona_id,
                            "persona_traits": persona_traits,
                            "system_instructions": system_instructions,
                            "user_prompt": user_prompt,
                            "max_output_tokens": scenario.max_output_tokens,
                        }
                    )

    plan_df = pd.DataFrame(rows)
    return plan_df


plan = build_generation_plan(
    scenario=SCENARIOS["neutral_main_t1"],
    prompts_df=selected_story_prompts,
    provider_name=PROVIDER_NAME,
    model_name=MODEL_NAME,
)

plan.shape, plan.head()

((150, 14),
                                          request_key    scenario_name  \
 0  0801434a3f3e596a98c9a1362eab0242402206bd40a3bd...  neutral_main_t1   
 1  02fff4ddbc2292c5f33756c293ab7fe398af068b3654a9...  neutral_main_t1   
 2  a7de0e4ed0b90477fbe75b7ba473dac7c00552bfcea651...  neutral_main_t1   
 3  a785ef7fee29265684eb034e4b7e5f86424f5d71f95159...  neutral_main_t1   
 4  c723c561138b07330b7b1901a1ba05752abe2d7adac5e6...  neutral_main_t1   
 
   provider    model condition_type  prompt_id  \
 0   openai  gpt-5.4        neutral      10491   
 1   openai  gpt-5.4        neutral      10491   
 2   openai  gpt-5.4        neutral      10491   
 3   openai  gpt-5.4        neutral      10491   
 4   openai  gpt-5.4        neutral      10491   
 
                                         story_prompt  temperature  run_idx  \
 0  A short Horror story . Something to chill the ...          1.0        0   
 1  A short Horror story . Something to chill the ...          1.0        1   
 2 

In [22]:
# !pip install -U tiktoken

import tiktoken
import pandas as pd

# Approximate tokenizer. If tiktoken does not know GPT-5.4 yet,
# o200k_base is a reasonable modern OpenAI tokenizer approximation.
try:
    enc = tiktoken.encoding_for_model("gpt-5.4")
except Exception:
    enc = tiktoken.get_encoding("o200k_base")


GPT54_BATCH_INPUT_PER_1M = 1.25
GPT54_BATCH_CACHED_INPUT_PER_1M = 0.125
GPT54_BATCH_OUTPUT_PER_1M = 7.50


def count_tokens(text):
    if pd.isna(text):
        return 0
    return len(enc.encode(str(text)))


def build_all_planned_requests(
    prompts_df,
    provider_name="openai",
    model_name="gpt-5.4",
):
    scenario_names = [
        "neutral_main_t1",
        "neutral_temperature_robustness",
        "personality_grid",
    ]

    plans = []
    for scenario_name in scenario_names:
        plan = build_generation_plan(
            scenario=SCENARIOS[scenario_name],
            prompts_df=prompts_df,
            provider_name=provider_name,
            model_name=model_name,
        )
        plans.append(plan)

    return pd.concat(plans, ignore_index=True)


all_plan_df = build_all_planned_requests(
    prompts_df=selected_story_prompts,
    provider_name="openai",
    model_name="gpt-5.4",
)

all_plan_df["system_tokens"] = all_plan_df["system_instructions"].map(count_tokens)
all_plan_df["user_tokens"] = all_plan_df["user_prompt"].map(count_tokens)
all_plan_df["estimated_input_tokens"] = all_plan_df["system_tokens"] + all_plan_df["user_tokens"]

print(f"Total planned requests: {len(all_plan_df):,}")
all_plan_df.groupby("scenario_name").agg(
    n_requests=("request_key", "size"),
    mean_input_tokens=("estimated_input_tokens", "mean"),
    median_input_tokens=("estimated_input_tokens", "median"),
    max_input_tokens=("estimated_input_tokens", "max"),
    total_input_tokens=("estimated_input_tokens", "sum"),
)

Total planned requests: 3,090


,n_requests,mean_input_tokens,median_input_tokens,max_input_tokens,total_input_tokens
scenario_name,,,,,
neutral_main_t1,150,67.0,62.0,78,10050
neutral_temperature_robustness,60,67.0,62.0,78,4020
personality_grid,2880,128.5,124.0,142,370080


In [23]:
def estimate_generation_cost(
    plan_df,
    assumed_output_tokens_per_request=250,
    input_price_per_1m=GPT54_BATCH_INPUT_PER_1M,
    output_price_per_1m=GPT54_BATCH_OUTPUT_PER_1M,
):
    n_requests = len(plan_df)
    input_tokens = int(plan_df["estimated_input_tokens"].sum())
    output_tokens = int(n_requests * assumed_output_tokens_per_request)

    input_cost = input_tokens / 1_000_000 * input_price_per_1m
    output_cost = output_tokens / 1_000_000 * output_price_per_1m
    total_cost = input_cost + output_cost

    return {
        "n_requests": n_requests,
        "estimated_input_tokens": input_tokens,
        "assumed_output_tokens": output_tokens,
        "input_cost_usd": input_cost,
        "output_cost_usd": output_cost,
        "total_cost_usd": total_cost,
    }


cost_scenarios = pd.DataFrame([
    {
        "assumed_output_tokens_per_request": out_tok,
        **estimate_generation_cost(all_plan_df, assumed_output_tokens_per_request=out_tok)
    }
    for out_tok in [150, 180, 250, 350, 500, 800]
])

cost_scenarios

,assumed_output_tokens_per_request,n_requests,estimated_input_tokens,assumed_output_tokens,input_cost_usd,output_cost_usd,total_cost_usd
0,150,3090,384150,463500,0.480187,3.47625,3.956438
1,180,3090,384150,556200,0.480187,4.17150,4.651687
2,250,3090,384150,772500,0.480187,5.79375,6.273937
3,350,3090,384150,1081500,0.480187,8.11125,8.591437
4,500,3090,384150,1545000,0.480187,11.58750,12.067687
5,800,3090,384150,2472000,0.480187,18.54000,19.020187


In [24]:
from openai import OpenAI

client = OpenAI()

BATCH_DIR = Path("ai_data/openai_batches")
BATCH_DIR.mkdir(parents=True, exist_ok=True)

In [25]:
# def make_openai_batch_input_jsonl(
#     scenario_name: str,
#     provider_name: str,
#     model_name: str,
#     prompts_df: pd.DataFrame,
#     output_dir: Path = BATCH_DIR,
#     exclude_existing_successes: bool = True,
# ) -> tuple[Path, Path, pd.DataFrame]:
#     """
#     Build a JSONL file suitable for OpenAI Batch API using /v1/responses.

#     Returns
#     -------
#     batch_jsonl_path:
#         JSONL file to upload to OpenAI.
#     plan_csv_path:
#         Local metadata plan for joining results back later.
#     batch_plan:
#         The subset of planned requests included in this batch.
#     """
#     scenario = SCENARIOS[scenario_name]

#     plan_df = build_generation_plan(
#         scenario=scenario,
#         prompts_df=prompts_df,
#         provider_name=provider_name,
#         model_name=model_name,
#     )

#     # If you have already downloaded/processed successful outputs for this scenario,
#     # skip those request_keys.
#     existing_output_path = AI_DATA_DIR / f"{provider_name}__{model_name}__{scenario_name}.jsonl"
#     if exclude_existing_successes and existing_output_path.exists():
#         done_keys = successful_request_keys(existing_output_path)
#         plan_df = plan_df[~plan_df["request_key"].isin(done_keys)].copy()

#     stem = f"{provider_name}__{model_name}__{scenario_name}"
#     batch_jsonl_path = output_dir / f"{stem}__batch_input.jsonl"
#     plan_csv_path = output_dir / f"{stem}__batch_plan.csv"

#     plan_df.to_csv(plan_csv_path, index=False)

#     with open(batch_jsonl_path, "w", encoding="utf-8") as f:
#         for _, row in plan_df.iterrows():
#             body = {
#                 "model": model_name,
#                 "instructions": row["system_instructions"],
#                 "input": row["user_prompt"],
#                 "temperature": float(row["temperature"]),
#                 "max_output_tokens": int(row["max_output_tokens"]),
#                 # For supported models, this can help keep repeated-prefix routing stable.
#                 # It will not help much if prompts are under 1024 tokens, but it is harmless.
#                 "prompt_cache_key": f"{scenario_name}:{row['condition_type']}:{row.get('persona_id') or 'neutral'}",
#             }

#             # Extended cache is available for GPT-5.4 and GPT-5.5; useful mostly if prompts are long enough.
#             if model_name.startswith(("gpt-5.4", "gpt-5.5")):
#                 body["prompt_cache_retention"] = "24h"

#             request = {
#                 "custom_id": row["request_key"],
#                 "method": "POST",
#                 "url": "/v1/responses",
#                 "body": body,
#             }
#             f.write(json.dumps(request, ensure_ascii=False) + "\n")

#     print(f"Batch requests written: {len(plan_df):,}")
#     print(f"Batch JSONL: {batch_jsonl_path}")
#     print(f"Plan CSV:    {plan_csv_path}")

#     return batch_jsonl_path, plan_csv_path, plan_df

In [77]:
def make_openai_batch_input_jsonl(
    scenario_name: str,
    provider_name: str,
    model_name: str,
    prompts_df: pd.DataFrame,
    output_dir: Path = BATCH_DIR,
    exclude_existing_successes: bool = True,
) -> tuple[Path, Path, pd.DataFrame]:
    """
    Build a JSONL file suitable for OpenAI Batch API using /v1/responses.

    This version intentionally does NOT include prompt_cache_key or
    prompt_cache_retention, because the personality IDs can exceed OpenAI's
    prompt_cache_key length limit and caching is unlikely to help for these
    short prompts anyway.
    """
    scenario = SCENARIOS[scenario_name]

    plan_df = build_generation_plan(
        scenario=scenario,
        prompts_df=prompts_df,
        provider_name=provider_name,
        model_name=model_name,
    )

    existing_output_path = AI_DATA_DIR / f"{provider_name}__{model_name}__{scenario_name}.jsonl"

    if exclude_existing_successes and existing_output_path.exists():
        done_keys = successful_request_keys(existing_output_path)
        plan_df = plan_df[~plan_df["request_key"].isin(done_keys)].copy()

    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"{provider_name}__{model_name}__{scenario_name}__{timestamp}"

    batch_jsonl_path = output_dir / f"{stem}__batch_input.jsonl"
    plan_csv_path = output_dir / f"{stem}__batch_plan.csv"

    plan_df.to_csv(plan_csv_path, index=False)

    with open(batch_jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            body = {
                "model": model_name,
                "instructions": row["system_instructions"],
                "input": row["user_prompt"],
                "temperature": float(row["temperature"]),
                "max_output_tokens": int(row["max_output_tokens"]),
            }

            request = {
                "custom_id": row["request_key"],
                "method": "POST",
                "url": "/v1/responses",
                "body": body,
            }

            f.write(json.dumps(request, ensure_ascii=False) + "\n")

    print(f"Batch requests written: {len(plan_df):,}")
    print(f"Batch JSONL: {batch_jsonl_path}")
    print(f"Plan CSV:    {plan_csv_path}")

    return batch_jsonl_path, plan_csv_path, plan_df

In [26]:
def submit_openai_batch(
    batch_jsonl_path: Path,
    scenario_name: str,
    model_name: str,
) -> dict:
    """
    Upload a batch JSONL file and submit it to OpenAI Batch API.
    """
    batch_input_file = client.files.create(
        file=open(batch_jsonl_path, "rb"),
        purpose="batch",
    )

    batch = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={
            "scenario_name": scenario_name,
            "model": model_name,
            "local_input_file": str(batch_jsonl_path),
        },
    )

    batch_info = {
        "batch_id": batch.id,
        "input_file_id": batch_input_file.id,
        "scenario_name": scenario_name,
        "model": model_name,
        "submitted_at_utc": now_iso(),
        "local_input_file": str(batch_jsonl_path),
    }

    manifest_path = batch_jsonl_path.with_suffix(".batch_manifest.json")
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(batch_info, f, indent=2)

    print("Submitted batch")
    print(json.dumps(batch_info, indent=2))

    return batch_info

In [27]:
MODEL_NAME = "gpt-5.4"
PROVIDER_NAME = "openai"

batch_jsonl_path, plan_csv_path, batch_plan = make_openai_batch_input_jsonl(
    scenario_name="neutral_main_t1",
    provider_name=PROVIDER_NAME,
    model_name=MODEL_NAME,
    prompts_df=selected_story_prompts,
)

batch_info = submit_openai_batch(
    batch_jsonl_path=batch_jsonl_path,
    scenario_name="neutral_main_t1",
    model_name=MODEL_NAME,
)

Batch requests written: 150
Batch JSONL: ai_data/openai_batches/openai__gpt-5.4__neutral_main_t1__batch_input.jsonl
Plan CSV:    ai_data/openai_batches/openai__gpt-5.4__neutral_main_t1__batch_plan.csv
Submitted batch
{
  "batch_id": "batch_69f033e30bf881908ae9fcc4753ff657",
  "input_file_id": "file-WPVzmdudsdQBj3RtG9f67S",
  "scenario_name": "neutral_main_t1",
  "model": "gpt-5.4",
  "submitted_at_utc": "2026-04-28T04:13:23.604656+00:00",
  "local_input_file": "ai_data/openai_batches/openai__gpt-5.4__neutral_main_t1__batch_input.jsonl"
}


In [34]:
def check_openai_batch(batch_id: str):
    batch = client.batches.retrieve(batch_id)
    print(batch)
    return batch


batch = check_openai_batch(batch_info["batch_id"])

Batch(id='batch_69f033e30bf881908ae9fcc4753ff657', completion_window='24h', created_at=1777349603, endpoint='/v1/responses', input_file_id='file-WPVzmdudsdQBj3RtG9f67S', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1777349966, error_file_id=None, errors=None, expired_at=None, expires_at=1777436003, failed_at=None, finalizing_at=1777349943, in_progress_at=1777349664, metadata={'scenario_name': 'neutral_main_t1', 'model': 'gpt-5.4', 'local_input_file': 'ai_data/openai_batches/openai__gpt-5.4__neutral_main_t1__batch_input.jsonl'}, model='gpt-5.4-2026-03-05', output_file_id='file-UCwsK7UbPwkA3f2jxJxBxH', request_counts=BatchRequestCounts(completed=150, failed=0, total=150), usage=BatchUsage(input_tokens=11550, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=21637, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=33187))


In [31]:
def download_openai_batch_results(
    batch_id: str,
    output_dir: Path = BATCH_DIR,
) -> tuple[Optional[Path], Optional[Path]]:
    batch = client.batches.retrieve(batch_id)

    if batch.status != "completed":
        print(f"Batch is not completed yet. Current status: {batch.status}")
        return None, None

    output_path = output_dir / f"{batch_id}__output.jsonl"
    error_path = output_dir / f"{batch_id}__errors.jsonl"

    if batch.output_file_id:
        file_response = client.files.content(batch.output_file_id)
        output_path.write_text(file_response.text, encoding="utf-8")
        print(f"Downloaded output: {output_path}")

    if batch.error_file_id:
        error_response = client.files.content(batch.error_file_id)
        error_path.write_text(error_response.text, encoding="utf-8")
        print(f"Downloaded errors: {error_path}")
    else:
        error_path = None

    return output_path, error_path

In [35]:
# output_path, error_path = download_openai_batch_results(batch_info["batch_id"])
output_path, error_path = download_openai_batch_results("batch_69f033e30bf881908ae9fcc4753ff657")

Downloaded output: ai_data/openai_batches/batch_69f033e30bf881908ae9fcc4753ff657__output.jsonl


In [36]:
def extract_text_from_responses_api_body(body: dict) -> str:
    """
    Extract output text from a /v1/responses response body.
    Handles common Responses API output shape.
    """
    if not isinstance(body, dict):
        return ""

    # Some SDK/API variants expose output_text directly.
    if body.get("output_text"):
        return str(body["output_text"]).strip()

    texts = []
    for item in body.get("output", []) or []:
        for content in item.get("content", []) or []:
            if content.get("type") in {"output_text", "text"} and "text" in content:
                texts.append(content["text"])

    return "\n".join(texts).strip()


def parse_openai_batch_output_to_standard_jsonl(
    batch_output_path: Path,
    plan_csv_path: Path,
    scenario_name: str,
    provider_name: str,
    model_name: str,
    standard_output_dir: Path = AI_DATA_DIR,
) -> Path:
    """
    Convert OpenAI Batch output JSONL into the same standard JSONL shape
    used by the synchronous runner.
    """
    plan_df = pd.read_csv(plan_csv_path)
    plan_by_key = {
        row["request_key"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    standard_output_path = standard_output_dir / f"{provider_name}__{model_name}__{scenario_name}.jsonl"

    batch_records = read_jsonl(batch_output_path)

    n_success = 0
    n_error = 0

    for rec in batch_records:
        request_key = rec.get("custom_id")
        plan_row = plan_by_key.get(request_key, {})

        response = rec.get("response")
        error = rec.get("error")

        if response and response.get("body"):
            body = response["body"]
            text = extract_text_from_responses_api_body(body)
            usage = body.get("usage")

            standard_record = {
                **plan_row,
                "created_at_utc": now_iso(),
                "status": "success" if text else "empty_text",
                "text": text,
                "provider_response_id": body.get("id"),
                "usage": usage,
                "raw_response": None,
                "error": None if text else "No text extracted from response body.",
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
            }
            n_success += int(bool(text))
            n_error += int(not bool(text))

        else:
            standard_record = {
                **plan_row,
                "created_at_utc": now_iso(),
                "status": "error",
                "text": None,
                "provider_response_id": None,
                "usage": None,
                "raw_response": None,
                "error": error,
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
            }
            n_error += 1

        append_jsonl(standard_output_path, standard_record)

    print(f"Parsed batch output to: {standard_output_path}")
    print(f"Success-ish text records: {n_success:,}")
    print(f"Errors/empty:             {n_error:,}")

    return standard_output_path

In [37]:
standard_jsonl_path = parse_openai_batch_output_to_standard_jsonl(
    batch_output_path=output_path,
    plan_csv_path=plan_csv_path,
    scenario_name="neutral_main_t1",
    provider_name=PROVIDER_NAME,
    model_name=MODEL_NAME,
)



Parsed batch output to: ai_data/story_generations/openai__gpt-5.4__neutral_main_t1.jsonl
Success-ish text records: 150
Errors/empty:             0


In [40]:
def load_generation_jsonl(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    records = read_jsonl(path)
    df = pd.DataFrame(records)
    return df

In [41]:
main_df = load_generation_jsonl(standard_jsonl_path)

print(main_df.shape)
print(main_df["status"].value_counts(dropna=False))

main_df[["prompt_id", "temperature", "run_idx", "status", "text"]].head()

(150, 23)
status
success    150
Name: count, dtype: int64


,prompt_id,temperature,run_idx,status,text
0,10491,1.0,0,success,"At 2:13 a.m., the baby monitor crackled awake...."
1,10491,1.0,1,success,"At 2:13 every night, the baby monitor crackled..."
2,10491,1.0,2,success,"At 2:13 a.m., the baby monitor clicked on by i..."
3,10491,1.0,3,success,"At 2:13 a.m., the baby monitor crackled.\n\n“M..."
4,10491,1.0,4,success,"At 2:13 a.m., the baby monitor crackled.\n\n“M..."


In [42]:
main_df["word_count"] = main_df["text"].fillna("").str.findall(r"\b[\w']+\b").str.len()

main_df.groupby("prompt_id").agg(
    n=("text", "size"),
    mean_words=("word_count", "mean"),
    median_words=("word_count", "median"),
    min_words=("word_count", "min"),
    max_words=("word_count", "max"),
)

,n,mean_words,median_words,min_words,max_words
prompt_id,,,,,
10491,50,101.08,102.0,79,113
93742,50,101.06,101.5,94,110
93855,50,116.48,117.0,110,121


In [43]:
PROVIDER_NAME = "openai"
MODEL_NAME = "gpt-5.4"

ROBUSTNESS_SCENARIOS = [
    "neutral_temperature_robustness",
    "personality_grid",
]

ROBUSTNESS_SCENARIOS

['neutral_temperature_robustness', 'personality_grid']

In [44]:
def submit_scenarios_as_batches(
    scenario_names,
    provider_name,
    model_name,
    prompts_df,
    batch_dir=BATCH_DIR,
    exclude_existing_successes=True,
):
    """
    Create, upload, and submit one OpenAI Batch job per scenario.

    Returns
    -------
    DataFrame with batch metadata.
    """
    submitted = []

    for scenario_name in scenario_names:
        print("=" * 100)
        print(f"Preparing scenario: {scenario_name}")
        print("=" * 100)

        batch_jsonl_path, plan_csv_path, batch_plan = make_openai_batch_input_jsonl(
            scenario_name=scenario_name,
            provider_name=provider_name,
            model_name=model_name,
            prompts_df=prompts_df,
            output_dir=batch_dir,
            exclude_existing_successes=exclude_existing_successes,
        )

        if len(batch_plan) == 0:
            print(f"No remaining requests for scenario: {scenario_name}")
            continue

        batch_info = submit_openai_batch(
            batch_jsonl_path=batch_jsonl_path,
            scenario_name=scenario_name,
            model_name=model_name,
        )

        batch_info["provider_name"] = provider_name
        batch_info["plan_csv_path"] = str(plan_csv_path)
        batch_info["n_requests_submitted"] = len(batch_plan)

        submitted.append(batch_info)

    submitted_df = pd.DataFrame(submitted)

    if len(submitted_df) > 0:
        manifest_path = batch_dir / f"{provider_name}__{model_name}__robustness_batch_manifest.csv"
        submitted_df.to_csv(manifest_path, index=False)
        print("\nSaved robustness batch manifest:")
        print(manifest_path)

    return submitted_df

In [45]:
submitted_robustness_batches = submit_scenarios_as_batches(
    scenario_names=ROBUSTNESS_SCENARIOS,
    provider_name=PROVIDER_NAME,
    model_name=MODEL_NAME,
    prompts_df=selected_story_prompts,
    exclude_existing_successes=True,
)

submitted_robustness_batches

Preparing scenario: neutral_temperature_robustness
Batch requests written: 60
Batch JSONL: ai_data/openai_batches/openai__gpt-5.4__neutral_temperature_robustness__batch_input.jsonl
Plan CSV:    ai_data/openai_batches/openai__gpt-5.4__neutral_temperature_robustness__batch_plan.csv
Submitted batch
{
  "batch_id": "batch_69f036e6a7948190899bcfbe9c980aa4",
  "input_file_id": "file-E4Bau1uEEpkgSfgyFag8LZ",
  "scenario_name": "neutral_temperature_robustness",
  "model": "gpt-5.4",
  "submitted_at_utc": "2026-04-28T04:26:14.804006+00:00",
  "local_input_file": "ai_data/openai_batches/openai__gpt-5.4__neutral_temperature_robustness__batch_input.jsonl"
}
Preparing scenario: personality_grid
Batch requests written: 2,880
Batch JSONL: ai_data/openai_batches/openai__gpt-5.4__personality_grid__batch_input.jsonl
Plan CSV:    ai_data/openai_batches/openai__gpt-5.4__personality_grid__batch_plan.csv
Submitted batch
{
  "batch_id": "batch_69f036e7a7d88190b09d43b0366a0c6f",
  "input_file_id": "file-D6FUU

,batch_id,input_file_id,scenario_name,model,submitted_at_utc,local_input_file,provider_name,plan_csv_path,n_requests_submitted
0,batch_69f036e6a7948190899bcfbe9c980aa4,file-E4Bau1uEEpkgSfgyFag8LZ,neutral_temperature_robustness,gpt-5.4,2026-04-28T04:26:14.804006+00:00,ai_data/openai_batches/openai__gpt-5.4__neutra...,openai,ai_data/openai_batches/openai__gpt-5.4__neutra...,60
1,batch_69f036e7a7d88190b09d43b0366a0c6f,file-D6FUU27BnVeBF8Q8xbB2T9,personality_grid,gpt-5.4,2026-04-28T04:26:15.811132+00:00,ai_data/openai_batches/openai__gpt-5.4__person...,openai,ai_data/openai_batches/openai__gpt-5.4__person...,2880


In [ ]:
## reloading code
# robustness_manifest_path = BATCH_DIR / f"{PROVIDER_NAME}__{MODEL_NAME}__robustness_batch_manifest.csv"

# submitted_robustness_batches = pd.read_csv(robustness_manifest_path)
# submitted_robustness_batches

In [62]:
def check_batches_from_manifest(batch_manifest_df):
    rows = []

    for _, row in batch_manifest_df.iterrows():
        batch_id = row["batch_id"]
        batch = client.batches.retrieve(batch_id)

        rows.append({
            "batch_id": batch.id,
            "scenario_name": row.get("scenario_name"),
            "model_requested": row.get("model"),
            "model_resolved": batch.model,
            "status": batch.status,
            "completed": batch.request_counts.completed if batch.request_counts else None,
            "failed": batch.request_counts.failed if batch.request_counts else None,
            "total": batch.request_counts.total if batch.request_counts else None,
            "input_tokens": batch.usage.input_tokens if batch.usage else None,
            "cached_input_tokens": batch.usage.input_tokens_details.cached_tokens if batch.usage else None,
            "output_tokens": batch.usage.output_tokens if batch.usage else None,
            "reasoning_tokens": batch.usage.output_tokens_details.reasoning_tokens if batch.usage else None,
            "total_tokens": batch.usage.total_tokens if batch.usage else None,
            "output_file_id": batch.output_file_id,
            "error_file_id": batch.error_file_id,
            "created_at": batch.created_at,
            "in_progress_at": batch.in_progress_at,
            "finalizing_at": batch.finalizing_at,
            "completed_at": batch.completed_at,
        })

    status_df = pd.DataFrame(rows)
    return status_df


robustness_status_df = check_batches_from_manifest(submitted_robustness_batches)
robustness_status_df

,batch_id,scenario_name,model_requested,model_resolved,status,completed,failed,total,input_tokens,cached_input_tokens,output_tokens,reasoning_tokens,total_tokens,output_file_id,error_file_id,created_at,in_progress_at,finalizing_at,completed_at
0,batch_69f036e6a7948190899bcfbe9c980aa4,neutral_temperature_robustness,gpt-5.4,gpt-5.4-2026-03-05,completed,60,0,60,4620,0,8661,0,13281,file-M65FB2bZxso6ewWxFuYZSA,None,1777350374,1777350376,1.777351e+09,1.777351e+09
1,batch_69f036e7a7d88190b09d43b0366a0c6f,personality_grid,gpt-5.4,gpt-5.4-2026-03-05,in_progress,0,2879,2880,0,0,0,0,0,None,None,1777350375,1777350377,NaN,NaN


In [73]:
personality_batch_id = "batch_69f036e7a7d88190b09d43b0366a0c6f"

batch = client.batches.retrieve(personality_batch_id)
print(batch)
print("status:", batch.status)
print("completed:", batch.request_counts.completed)
print("failed:", batch.request_counts.failed)
print("total:", batch.request_counts.total)
print("output_file_id:", batch.output_file_id)
print("error_file_id:", batch.error_file_id)

Batch(id='batch_69f036e7a7d88190b09d43b0366a0c6f', completion_window='24h', created_at=1777350375, endpoint='/v1/responses', input_file_id='file-D6FUU27BnVeBF8Q8xbB2T9', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1777352397, error_file_id='file-KwFmQ2mRzMp9d3R6GyMWoz', errors=None, expired_at=None, expires_at=1777436775, failed_at=None, finalizing_at=1777352260, in_progress_at=1777350377, metadata={'scenario_name': 'personality_grid', 'model': 'gpt-5.4', 'local_input_file': 'ai_data/openai_batches/openai__gpt-5.4__personality_grid__batch_input.jsonl'}, model='gpt-5.4-2026-03-05', output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=2880, total=2880), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0))
status: completed
completed: 0
failed: 2880
total: 2880
output_file_id: None
error_fil

In [74]:
from pathlib import Path

error_file_id = client.batches.retrieve(personality_batch_id).error_file_id

if error_file_id is None:
    print("No error_file_id yet. Recheck batch status in a minute.")
else:
    error_response = client.files.content(error_file_id)
    error_path = BATCH_DIR / f"{personality_batch_id}__errors.jsonl"
    error_path.write_text(error_response.text, encoding="utf-8")
    print(f"Saved errors to: {error_path}")

Saved errors to: ai_data/openai_batches/batch_69f036e7a7d88190b09d43b0366a0c6f__errors.jsonl


In [75]:
personality_error_path = BATCH_DIR / f"{personality_batch_id}__errors.jsonl"

errors = read_jsonl(personality_error_path)
print(f"Number of error records: {len(errors):,}")

for e in errors[:5]:
    print("=" * 100)
    print(json.dumps(e, indent=2)[:4000])

Number of error records: 2,880
{
  "id": "batch_req_69f03e454e3c8190b39b256210884034",
  "custom_id": "ae45daed1437f436a96479ff43b7d13b27c43d776049e76f39843115494b3d22",
  "response": {
    "status_code": 400,
    "request_id": "3381acec-c0d4-4b88-b9ce-316171062d4b",
    "body": {
      "error": {
        "message": "Invalid 'prompt_cache_key': string too long. Expected a string with maximum length 64, but got a string with length 96 instead.",
        "type": "invalid_request_error",
        "param": "prompt_cache_key",
        "code": "string_above_max_length"
      }
    }
  },
  "error": null
}
{
  "id": "batch_req_69f03e45500c819097128c5eca98ca18",
  "custom_id": "c36172c6b64175f51efb2f996fc4fb9de90f4b4c6675cfb773b11db36d804a50",
  "response": {
    "status_code": 400,
    "request_id": "3184c541-71e5-4a2b-ad2c-986ae7868ab6",
    "body": {
      "error": {
        "message": "Invalid 'prompt_cache_key': string too long. Expected a string with maximum length 64, but got a string wi

In [76]:
def extract_batch_error_message(error_record):
    err = error_record.get("error")
    if isinstance(err, dict):
        return err.get("message") or err.get("code") or str(err)
    return str(err)

error_summary = (
    pd.Series([extract_batch_error_message(e) for e in errors])
    .value_counts()
    .reset_index()
)

error_summary.columns = ["error_message", "count"]
error_summary.head(20)

,error_message,count
0,None,2880


In [78]:
neutral_temp_batch_id = "batch_69f036e6a7948190899bcfbe9c980aa4"

neutral_temp_output_path, neutral_temp_error_path = download_openai_batch_results(
    neutral_temp_batch_id,
    output_dir=BATCH_DIR,
)

neutral_temp_output_path, neutral_temp_error_path

Downloaded output: ai_data/openai_batches/batch_69f036e6a7948190899bcfbe9c980aa4__output.jsonl


(PosixPath('ai_data/openai_batches/batch_69f036e6a7948190899bcfbe9c980aa4__output.jsonl'),
 None)

In [79]:
neutral_temp_plan_csv_path = Path(
    "ai_data/openai_batches/openai__gpt-5.4__neutral_temperature_robustness__batch_plan.csv"
)

neutral_temp_standard_jsonl_path = parse_openai_batch_output_to_standard_jsonl(
    batch_output_path=neutral_temp_output_path,
    plan_csv_path=neutral_temp_plan_csv_path,
    scenario_name="neutral_temperature_robustness",
    provider_name="openai",
    model_name="gpt-5.4",
)

neutral_temp_df = load_generation_jsonl(neutral_temp_standard_jsonl_path)

print(neutral_temp_df.shape)
print(neutral_temp_df["status"].value_counts(dropna=False))
neutral_temp_df[["scenario_name", "prompt_id", "temperature", "run_idx", "status", "text"]].head()

Parsed batch output to: ai_data/story_generations/openai__gpt-5.4__neutral_temperature_robustness.jsonl
Success-ish text records: 60
Errors/empty:             0
(60, 23)
status
success    60
Name: count, dtype: int64


,scenario_name,prompt_id,temperature,run_idx,status,text
0,neutral_temperature_robustness,10491,0.7,0,success,"Every night, the baby monitor crackled at 3:07..."
1,neutral_temperature_robustness,10491,0.7,1,success,"At 2:13 a.m., my daughter padded into my room,..."
2,neutral_temperature_robustness,10491,0.7,2,success,"At 2:13 a.m., the baby monitor crackled.\n\nMy..."
3,neutral_temperature_robustness,10491,0.7,3,success,"At 2:13 every night, the baby monitor crackled..."
4,neutral_temperature_robustness,10491,0.7,4,success,"At 2:13 a.m., the baby monitor crackled.\n\n“D..."


In [81]:
def load_all_generation_files(
    provider_name: str,
    model_name: str,
    output_dir: Path = AI_DATA_DIR,
) -> pd.DataFrame:
    paths = sorted(output_dir.glob(f"{provider_name}__{model_name}__*.jsonl"))
    paths = [p for p in paths if "__plan" not in p.name]

    dfs = []
    for p in paths:
        df = load_generation_jsonl(p)
        df["source_file"] = str(p)
        dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)

In [83]:
def deduplicate_generation_records(df):
    """
    Deduplicate by request_key.

    Priority:
    1. success
    2. empty_text
    3. error

    Within same priority, keep the last occurrence.
    """
    df = df.copy()

    status_priority = {
        "success": 3,
        "empty_text": 2,
        "error": 1,
    }

    df["_status_priority"] = df["status"].map(status_priority).fillna(0)
    df["_original_order"] = range(len(df))

    df = (
        df.sort_values(["request_key", "_status_priority", "_original_order"])
          .drop_duplicates("request_key", keep="last")
          .drop(columns=["_status_priority", "_original_order"])
          .reset_index(drop=True)
    )

    return df

In [84]:
all_ai_story_df = load_all_generation_files(
    provider_name="openai",
    model_name="gpt-5.4",
    output_dir=AI_DATA_DIR,
)

all_ai_story_df_dedup = deduplicate_generation_records(all_ai_story_df)

print("Before dedup:", all_ai_story_df.shape)
print("After dedup: ", all_ai_story_df_dedup.shape)
print(all_ai_story_df_dedup["scenario_name"].value_counts(dropna=False))

Before dedup: (210, 24)
After dedup:  (210, 24)
scenario_name
neutral_main_t1                   150
neutral_temperature_robustness     60
Name: count, dtype: int64


In [85]:
personality_jsonl_path, personality_plan_csv_path, personality_plan = make_openai_batch_input_jsonl(
    scenario_name="personality_grid",
    provider_name=PROVIDER_NAME,
    model_name=MODEL_NAME,
    prompts_df=selected_story_prompts,
    output_dir=BATCH_DIR,
    exclude_existing_successes=True,
)

print(personality_plan.shape)
personality_plan.head()

Batch requests written: 2,880
Batch JSONL: ai_data/openai_batches/openai__gpt-5.4__personality_grid__20260428_104110__batch_input.jsonl
Plan CSV:    ai_data/openai_batches/openai__gpt-5.4__personality_grid__20260428_104110__batch_plan.csv
(2880, 14)


,request_key,scenario_name,provider,model,condition_type,prompt_id,story_prompt,temperature,run_idx,persona_id,persona_traits,system_instructions,user_prompt,max_output_tokens
0,ae45daed1437f436a96479ff43b7d13b27c43d776049e7...,personality_grid,openai,gpt-5.4,personality,10491,A short Horror story . Something to chill the ...,0.7,0,extroverted__agreeable__conscientious__neuroti...,"[extroverted, agreeable, conscientious, neurot...",You are participating in a creative writing ta...,Prompt:\nA short Horror story . Something to c...,800
1,c36172c6b64175f51efb2f996fc4fb9de90f4b4c6675cf...,personality_grid,openai,gpt-5.4,personality,10491,A short Horror story . Something to chill the ...,0.7,1,extroverted__agreeable__conscientious__neuroti...,"[extroverted, agreeable, conscientious, neurot...",You are participating in a creative writing ta...,Prompt:\nA short Horror story . Something to c...,800
2,575a1127db9ae16e580f42f5b608acb90b28506431f3ec...,personality_grid,openai,gpt-5.4,personality,10491,A short Horror story . Something to chill the ...,0.7,2,extroverted__agreeable__conscientious__neuroti...,"[extroverted, agreeable, conscientious, neurot...",You are participating in a creative writing ta...,Prompt:\nA short Horror story . Something to c...,800
3,1cf3157b3220bc7d807dc2cb1bdaaaa1a50aaf9be71a29...,personality_grid,openai,gpt-5.4,personality,10491,A short Horror story . Something to chill the ...,0.7,3,extroverted__agreeable__conscientious__neuroti...,"[extroverted, agreeable, conscientious, neurot...",You are participating in a creative writing ta...,Prompt:\nA short Horror story . Something to c...,800
4,0522330445f8e1db4bbc5750f893efa67a3fd9677c53df...,personality_grid,openai,gpt-5.4,personality,10491,A short Horror story . Something to chill the ...,0.7,4,extroverted__agreeable__conscientious__neuroti...,"[extroverted, agreeable, conscientious, neurot...",You are participating in a creative writing ta...,Prompt:\nA short Horror story . Something to c...,800


In [86]:
personality_batch_info_fixed = submit_openai_batch(
    batch_jsonl_path=personality_jsonl_path,
    scenario_name="personality_grid",
    model_name=MODEL_NAME,
)

personality_batch_info_fixed

Submitted batch
{
  "batch_id": "batch_69f0c70d76f88190832f07d6b271996a",
  "input_file_id": "file-GwBu4ciCW3FojgKjjA4VYd",
  "scenario_name": "personality_grid",
  "model": "gpt-5.4",
  "submitted_at_utc": "2026-04-28T14:41:17.867487+00:00",
  "local_input_file": "ai_data/openai_batches/openai__gpt-5.4__personality_grid__20260428_104110__batch_input.jsonl"
}


{'batch_id': 'batch_69f0c70d76f88190832f07d6b271996a',
 'input_file_id': 'file-GwBu4ciCW3FojgKjjA4VYd',
 'scenario_name': 'personality_grid',
 'model': 'gpt-5.4',
 'submitted_at_utc': '2026-04-28T14:41:17.867487+00:00',
 'local_input_file': 'ai_data/openai_batches/openai__gpt-5.4__personality_grid__20260428_104110__batch_input.jsonl'}

In [107]:
fixed_personality_batch_id = "batch_69f0c70d76f88190832f07d6b271996a"

batch = check_openai_batch(fixed_personality_batch_id)

Batch(id='batch_69f0c70d76f88190832f07d6b271996a', completion_window='24h', created_at=1777387277, endpoint='/v1/responses', input_file_id='file-GwBu4ciCW3FojgKjjA4VYd', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1777414312, error_file_id='file-GAX1qE22X1Qj8VzZD2tehA', errors=None, expired_at=None, expires_at=1777473677, failed_at=None, finalizing_at=1777414108, in_progress_at=1777387339, metadata={'scenario_name': 'personality_grid', 'model': 'gpt-5.4', 'local_input_file': 'ai_data/openai_batches/openai__gpt-5.4__personality_grid__20260428_104110__batch_input.jsonl'}, model='gpt-5.4-2026-03-05', output_file_id='file-W7GgG8hATTUhGSAVLhCW62', request_counts=BatchRequestCounts(completed=2703, failed=177, total=2880), usage=BatchUsage(input_tokens=374099, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=388034, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=762133))


In [108]:
personality_output_path, personality_error_path = download_openai_batch_results(
    fixed_personality_batch_id,
    output_dir=BATCH_DIR,
)

personality_output_path, personality_error_path

Downloaded output: ai_data/openai_batches/batch_69f0c70d76f88190832f07d6b271996a__output.jsonl
Downloaded errors: ai_data/openai_batches/batch_69f0c70d76f88190832f07d6b271996a__errors.jsonl


(PosixPath('ai_data/openai_batches/batch_69f0c70d76f88190832f07d6b271996a__output.jsonl'),
 PosixPath('ai_data/openai_batches/batch_69f0c70d76f88190832f07d6b271996a__errors.jsonl'))

In [109]:
personality_plan_csv_path = Path(
    "ai_data/openai_batches/openai__gpt-5.4__personality_grid__20260428_104110__batch_plan.csv"
)

personality_standard_jsonl_path = parse_openai_batch_output_to_standard_jsonl(
    batch_output_path=personality_output_path,
    plan_csv_path=personality_plan_csv_path,
    scenario_name="personality_grid",
    provider_name="openai",
    model_name="gpt-5.4",
)

personality_df = load_generation_jsonl(personality_standard_jsonl_path)

print(personality_df.shape)
print(personality_df["status"].value_counts(dropna=False))
personality_df[["scenario_name", "prompt_id", "temperature", "persona_id", "run_idx", "status", "text"]].head()

Parsed batch output to: ai_data/story_generations/openai__gpt-5.4__personality_grid.jsonl
Success-ish text records: 2,703
Errors/empty:             0
(2703, 23)
status
success    2703
Name: count, dtype: int64


,scenario_name,prompt_id,temperature,persona_id,run_idx,status,text
0,personality_grid,10491,0.7,extroverted__agreeable__conscientious__neuroti...,0,success,"At the housewarming, everyone laughed when I f..."
1,personality_grid,10491,0.7,extroverted__agreeable__conscientious__neuroti...,1,success,"At the party, everyone laughed when I found my..."
2,personality_grid,10491,0.7,extroverted__agreeable__conscientious__neuroti...,2,success,"At the party, everyone loved the old mirror in..."
3,personality_grid,10491,0.7,extroverted__agreeable__conscientious__neuroti...,3,success,"At the party, everyone laughed when I found a ..."
4,personality_grid,10491,0.7,extroverted__agreeable__conscientious__neuroti...,4,success,"At the party, everyone laughed when the old ho..."


In [110]:
personality_errors = read_jsonl(personality_error_path)

print(f"Number of failed records: {len(personality_errors):,}")

for e in personality_errors[:5]:
    print("=" * 100)
    print(json.dumps(e, indent=2)[:4000])

Number of failed records: 177
{
  "id": "batch_req_69f12fddef9c8190bcdfbcff31081eb3",
  "custom_id": "c56410fcdae8ef12aed7494375df09bab60625b6a487257e5f0bdeb31caf2a30",
  "response": {
    "status_code": 503,
    "request_id": "0920eea3-bfb6-40dd-b1d7-2efe70bae01e",
    "body": {
      "error": {
        "message": "Our servers are currently overloaded. Please try again later.",
        "type": "service_unavailable_error",
        "param": null,
        "code": "server_is_overloaded"
      }
    }
  },
  "error": null
}
{
  "id": "batch_req_69f12fdf40f8819089af5a7fffe23abf",
  "custom_id": "061a32ef24cfc0ffd86fff82ee44b3ae9b3c63259dd8681332b92e0e99c79dad",
  "response": {
    "status_code": 503,
    "request_id": "da2e4e86-5e62-4bbb-b883-d62fcdef458c",
    "body": {
      "error": {
        "message": "Our servers are currently overloaded. Please try again later.",
        "type": "service_unavailable_error",
        "param": null,
        "code": "server_is_overloaded"
      }
    }
 

In [111]:
def extract_error_message_from_batch_record(error_record):
    # Batch error can appear either in response.body.error or in top-level error
    response = error_record.get("response") or {}
    body = response.get("body") or {}
    err = body.get("error") or error_record.get("error")

    if isinstance(err, dict):
        return err.get("message") or err.get("code") or str(err)
    return str(err)

error_summary = (
    pd.Series([extract_error_message_from_batch_record(e) for e in personality_errors])
    .value_counts()
    .reset_index()
)

error_summary.columns = ["error_message", "count"]
error_summary.head(20)

,error_message,count
0,Our servers are currently overloaded. Please t...,177


In [112]:
all_ai_story_df = load_all_generation_files(
    provider_name="openai",
    model_name="gpt-5.4",
    output_dir=AI_DATA_DIR,
)

all_ai_story_df_dedup = deduplicate_generation_records(all_ai_story_df)

print("Before dedup:", all_ai_story_df.shape)
print("After dedup: ", all_ai_story_df_dedup.shape)
print(all_ai_story_df_dedup["scenario_name"].value_counts(dropna=False))

Before dedup: (2913, 24)
After dedup:  (2913, 24)
scenario_name
personality_grid                  2703
neutral_main_t1                    150
neutral_temperature_robustness      60
Name: count, dtype: int64


In [113]:
personality_coverage = (
    all_ai_story_df_dedup
    .query("scenario_name == 'personality_grid'")
    .groupby(["prompt_id", "temperature", "persona_id"], dropna=False)
    .agg(
        n_records=("request_key", "size"),
        n_success=("status", lambda x: (x == "success").sum()),
    )
    .reset_index()
)

incomplete_personality_cells = personality_coverage.query("n_success < 10")

print(f"Incomplete personality cells: {len(incomplete_personality_cells)}")
incomplete_personality_cells.head(30)

Incomplete personality cells: 126


,prompt_id,temperature,persona_id,n_records,n_success
1,10491,0.7,extroverted__agreeable__conscientious__emotion...,9,9
3,10491,0.7,extroverted__agreeable__conscientious__neuroti...,9,9
13,10491,0.7,extroverted__antagonistic__unconscientious__em...,9,9
14,10491,0.7,extroverted__antagonistic__unconscientious__ne...,9,9
16,10491,0.7,introverted__agreeable__conscientious__emotion...,9,9
17,10491,0.7,introverted__agreeable__conscientious__emotion...,9,9
18,10491,0.7,introverted__agreeable__conscientious__neuroti...,9,9
19,10491,0.7,introverted__agreeable__conscientious__neuroti...,9,9
22,10491,0.7,introverted__agreeable__unconscientious__neuro...,9,9
23,10491,0.7,introverted__agreeable__unconscientious__neuro...,9,9


In [114]:
backfill_jsonl_path, backfill_plan_csv_path, backfill_plan = make_openai_batch_input_jsonl(
    scenario_name="personality_grid",
    provider_name="openai",
    model_name="gpt-5.4",
    prompts_df=selected_story_prompts,
    output_dir=BATCH_DIR,
    exclude_existing_successes=True,
)

print(backfill_plan.shape)
backfill_plan[["prompt_id", "temperature", "persona_id", "run_idx"]].head()

Batch requests written: 177
Batch JSONL: ai_data/openai_batches/openai__gpt-5.4__personality_grid__20260428_200812__batch_input.jsonl
Plan CSV:    ai_data/openai_batches/openai__gpt-5.4__personality_grid__20260428_200812__batch_plan.csv
(177, 14)


,prompt_id,temperature,persona_id,run_idx
6,10491,0.7,extroverted__agreeable__conscientious__neuroti...,6
21,10491,0.7,extroverted__agreeable__conscientious__emotion...,1
136,10491,0.7,extroverted__antagonistic__unconscientious__ne...,6
142,10491,0.7,extroverted__antagonistic__unconscientious__em...,2
160,10491,0.7,introverted__agreeable__conscientious__neuroti...,0


In [115]:
backfill_batch_info = submit_openai_batch(
    batch_jsonl_path=backfill_jsonl_path,
    scenario_name="personality_grid",
    model_name="gpt-5.4",
)

backfill_batch_info

Submitted batch
{
  "batch_id": "batch_69f14bf599708190a7844fa167bced77",
  "input_file_id": "file-WNEPkhzzt7jpepdq4AYZa5",
  "scenario_name": "personality_grid",
  "model": "gpt-5.4",
  "submitted_at_utc": "2026-04-29T00:08:22.433614+00:00",
  "local_input_file": "ai_data/openai_batches/openai__gpt-5.4__personality_grid__20260428_200812__batch_input.jsonl"
}


{'batch_id': 'batch_69f14bf599708190a7844fa167bced77',
 'input_file_id': 'file-WNEPkhzzt7jpepdq4AYZa5',
 'scenario_name': 'personality_grid',
 'model': 'gpt-5.4',
 'submitted_at_utc': '2026-04-29T00:08:22.433614+00:00',
 'local_input_file': 'ai_data/openai_batches/openai__gpt-5.4__personality_grid__20260428_200812__batch_input.jsonl'}

In [118]:
backfill_batch_id = "batch_69f14bf599708190a7844fa167bced77"

batch = check_openai_batch(backfill_batch_id)

Batch(id='batch_69f14bf599708190a7844fa167bced77', completion_window='24h', created_at=1777421301, endpoint='/v1/responses', input_file_id='file-WNEPkhzzt7jpepdq4AYZa5', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1777422490, error_file_id=None, errors=None, expired_at=None, expires_at=1777507701, failed_at=None, finalizing_at=1777422482, in_progress_at=1777421364, metadata={'scenario_name': 'personality_grid', 'model': 'gpt-5.4', 'local_input_file': 'ai_data/openai_batches/openai__gpt-5.4__personality_grid__20260428_200812__batch_input.jsonl'}, model='gpt-5.4-2026-03-05', output_file_id='file-7j66dBcWniAaSg7JBVe9m8', request_counts=BatchRequestCounts(completed=177, failed=0, total=177), usage=BatchUsage(input_tokens=24781, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=26024, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=50805))


In [119]:
backfill_output_path, backfill_error_path = download_openai_batch_results(
    backfill_batch_id,
    output_dir=BATCH_DIR,
)

backfill_plan_csv_path = Path(
    "ai_data/openai_batches/openai__gpt-5.4__personality_grid__20260428_200812__batch_plan.csv"
)

backfill_standard_jsonl_path = parse_openai_batch_output_to_standard_jsonl(
    batch_output_path=backfill_output_path,
    plan_csv_path=backfill_plan_csv_path,
    scenario_name="personality_grid",
    provider_name="openai",
    model_name="gpt-5.4",
)

backfill_df = load_generation_jsonl(backfill_standard_jsonl_path)

print(backfill_df.shape)
print(backfill_df["status"].value_counts(dropna=False))

Downloaded output: ai_data/openai_batches/batch_69f14bf599708190a7844fa167bced77__output.jsonl
Parsed batch output to: ai_data/story_generations/openai__gpt-5.4__personality_grid.jsonl
Success-ish text records: 177
Errors/empty:             0
(2880, 23)
status
success    2880
Name: count, dtype: int64


In [120]:
all_ai_story_df = load_all_generation_files(
    provider_name="openai",
    model_name="gpt-5.4",
    output_dir=AI_DATA_DIR,
)

all_ai_story_df_dedup = deduplicate_generation_records(all_ai_story_df)

print("Before dedup:", all_ai_story_df.shape)
print("After dedup: ", all_ai_story_df_dedup.shape)
print(all_ai_story_df_dedup["scenario_name"].value_counts(dropna=False))

Before dedup: (3090, 24)
After dedup:  (3090, 24)
scenario_name
personality_grid                  2880
neutral_main_t1                    150
neutral_temperature_robustness      60
Name: count, dtype: int64


In [121]:
personality_coverage = (
    all_ai_story_df_dedup
    .query("scenario_name == 'personality_grid'")
    .groupby(["prompt_id", "temperature", "persona_id"], dropna=False)
    .agg(
        n_records=("request_key", "size"),
        n_success=("status", lambda x: (x == "success").sum()),
    )
    .reset_index()
)

incomplete_personality_cells = personality_coverage.query("n_success < 10")

print(f"Incomplete personality cells: {len(incomplete_personality_cells)}")
incomplete_personality_cells.head(30)

Incomplete personality cells: 0


,prompt_id,temperature,persona_id,n_records,n_success


In [122]:
CLEAN_AI_DIR = Path("ai_data/processed")
CLEAN_AI_DIR.mkdir(parents=True, exist_ok=True)

clean_path_pkl = CLEAN_AI_DIR / "openai__gpt-5.4__story_generations_all_scenarios.pkl"
clean_path_csv = CLEAN_AI_DIR / "openai__gpt-5.4__story_generations_all_scenarios.csv"

all_ai_story_df_dedup.to_pickle(clean_path_pkl)
all_ai_story_df_dedup.to_csv(clean_path_csv, index=False)

print(clean_path_pkl)
print(clean_path_csv)

ai_data/processed/openai__gpt-5.4__story_generations_all_scenarios.pkl
ai_data/processed/openai__gpt-5.4__story_generations_all_scenarios.csv


# Claude

In [123]:
!pip install -U anthropic pandas tqdm python-dotenv

In [124]:
import os

os.environ["ANTHROPIC_API_KEY"] = ""


In [125]:
from pathlib import Path
import json
import time
import datetime as dt
import pandas as pd
import re
import hashlib

import anthropic

anthropic_client = anthropic.Anthropic()

ANTHROPIC_BATCH_DIR = Path("ai_data/anthropic_batches")
ANTHROPIC_BATCH_DIR.mkdir(parents=True, exist_ok=True)

AI_DATA_DIR = Path("ai_data/story_generations")
AI_DATA_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_AI_DIR = Path("ai_data/processed")
CLEAN_AI_DIR.mkdir(parents=True, exist_ok=True)

In [126]:
PROVIDER_NAME = "anthropic"
MODEL_NAME = "claude-sonnet-4-5"  # primary defensible Claude model

In [127]:
def make_anthropic_custom_id(request_key: str) -> str:
    """
    Anthropic custom_id must be 1-64 chars and match ^[a-zA-Z0-9_-]{1,64}$.
    """
    return "r_" + request_key[:48]

In [128]:
def build_anthropic_generation_plan(
    scenario_name: str,
    model_name: str,
    prompts_df: pd.DataFrame,
) -> pd.DataFrame:
    scenario = SCENARIOS[scenario_name]

    plan_df = build_generation_plan(
        scenario=scenario,
        prompts_df=prompts_df,
        provider_name="anthropic",
        model_name=model_name,
    ).copy()

    plan_df["anthropic_custom_id"] = plan_df["request_key"].map(make_anthropic_custom_id)

    if plan_df["anthropic_custom_id"].duplicated().any():
        raise ValueError("Duplicate anthropic_custom_id found. Increase hash length.")

    return plan_df

In [129]:
def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def read_jsonl(path: Path):
    if not path.exists():
        return []
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def append_jsonl(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def successful_request_keys(path: Path) -> set:
    records = read_jsonl(path)
    return {
        r["request_key"]
        for r in records
        if r.get("status") == "success" and r.get("text")
    }

In [130]:
def make_anthropic_batch_requests(plan_df: pd.DataFrame) -> list:
    requests = []

    for _, row in plan_df.iterrows():
        params = {
            "model": row["model"],
            "max_tokens": int(row["max_output_tokens"]),
            "temperature": float(row["temperature"]),
            "system": row["system_instructions"],
            "messages": [
                {
                    "role": "user",
                    "content": row["user_prompt"],
                }
            ],
        }

        # Do NOT add cache_control for this short-prompt benchmark.
        # The prompt is below Claude's cacheable-token threshold.

        requests.append(
            {
                "custom_id": row["anthropic_custom_id"],
                "params": params,
            }
        )

    return requests


def submit_anthropic_batch(
    scenario_name: str,
    model_name: str,
    prompts_df: pd.DataFrame,
    output_dir: Path = ANTHROPIC_BATCH_DIR,
    exclude_existing_successes: bool = True,
) -> dict:
    standard_output_path = AI_DATA_DIR / f"anthropic__{model_name}__{scenario_name}.jsonl"

    plan_df = build_anthropic_generation_plan(
        scenario_name=scenario_name,
        model_name=model_name,
        prompts_df=prompts_df,
    )

    if exclude_existing_successes and standard_output_path.exists():
        done_keys = successful_request_keys(standard_output_path)
        plan_df = plan_df[~plan_df["request_key"].isin(done_keys)].copy()

    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"anthropic__{model_name}__{scenario_name}__{timestamp}"

    plan_csv_path = output_dir / f"{stem}__batch_plan.csv"
    request_json_path = output_dir / f"{stem}__batch_requests.json"

    plan_df.to_csv(plan_csv_path, index=False)

    requests = make_anthropic_batch_requests(plan_df)

    with open(request_json_path, "w", encoding="utf-8") as f:
        json.dump(requests, f, indent=2, ensure_ascii=False)

    print(f"Scenario: {scenario_name}")
    print(f"Requests to submit: {len(requests):,}")
    print(f"Plan CSV: {plan_csv_path}")
    print(f"Request JSON: {request_json_path}")

    if len(requests) == 0:
        return {
            "scenario_name": scenario_name,
            "model": model_name,
            "n_requests_submitted": 0,
            "plan_csv_path": str(plan_csv_path),
            "request_json_path": str(request_json_path),
            "batch_id": None,
        }

    batch = anthropic_client.messages.batches.create(
        requests=requests
    )

    batch_info = {
        "batch_id": batch.id,
        "scenario_name": scenario_name,
        "model": model_name,
        "submitted_at_utc": now_iso(),
        "n_requests_submitted": len(requests),
        "plan_csv_path": str(plan_csv_path),
        "request_json_path": str(request_json_path),
        "processing_status": getattr(batch, "processing_status", None),
    }

    manifest_path = output_dir / f"{stem}__batch_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(batch_info, f, indent=2)

    print("Submitted Anthropic batch:")
    print(json.dumps(batch_info, indent=2))

    return batch_info

In [131]:
claude_main_batch = submit_anthropic_batch(
    scenario_name="neutral_main_t1",
    model_name=MODEL_NAME,
    prompts_df=selected_story_prompts,
    exclude_existing_successes=True,
)

claude_main_batch

Scenario: neutral_main_t1
Requests to submit: 150
Plan CSV: ai_data/anthropic_batches/anthropic__claude-sonnet-4-5__neutral_main_t1__20260428_221900__batch_plan.csv
Request JSON: ai_data/anthropic_batches/anthropic__claude-sonnet-4-5__neutral_main_t1__20260428_221900__batch_requests.json
Submitted Anthropic batch:
{
  "batch_id": "msgbatch_01CVybKyD2VB9giNLvjkteid",
  "scenario_name": "neutral_main_t1",
  "model": "claude-sonnet-4-5",
  "submitted_at_utc": "2026-04-29T02:19:01.290674+00:00",
  "n_requests_submitted": 150,
  "plan_csv_path": "ai_data/anthropic_batches/anthropic__claude-sonnet-4-5__neutral_main_t1__20260428_221900__batch_plan.csv",
  "request_json_path": "ai_data/anthropic_batches/anthropic__claude-sonnet-4-5__neutral_main_t1__20260428_221900__batch_requests.json",
  "processing_status": "in_progress"
}


{'batch_id': 'msgbatch_01CVybKyD2VB9giNLvjkteid',
 'scenario_name': 'neutral_main_t1',
 'model': 'claude-sonnet-4-5',
 'submitted_at_utc': '2026-04-29T02:19:01.290674+00:00',
 'n_requests_submitted': 150,
 'plan_csv_path': 'ai_data/anthropic_batches/anthropic__claude-sonnet-4-5__neutral_main_t1__20260428_221900__batch_plan.csv',
 'request_json_path': 'ai_data/anthropic_batches/anthropic__claude-sonnet-4-5__neutral_main_t1__20260428_221900__batch_requests.json',
 'processing_status': 'in_progress'}

In [133]:
def check_anthropic_batch(batch_id: str):
    batch = anthropic_client.messages.batches.retrieve(batch_id)
    print(batch)
    return batch


batch = check_anthropic_batch(claude_main_batch["batch_id"])

MessageBatch(id='msgbatch_01CVybKyD2VB9giNLvjkteid', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 4, 29, 2, 19, 1, 180704, tzinfo=datetime.timezone.utc), ended_at=datetime.datetime(2026, 4, 29, 2, 20, 13, 517323, tzinfo=TzInfo(UTC)), expires_at=datetime.datetime(2026, 4, 30, 2, 19, 1, 180704, tzinfo=datetime.timezone.utc), processing_status='ended', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=150), results_url='https://api.anthropic.com/v1/messages/batches/msgbatch_01CVybKyD2VB9giNLvjkteid/results', type='message_batch')


In [134]:
def extract_text_from_anthropic_message(message) -> str:
    texts = []

    for block in getattr(message, "content", []) or []:
        block_type = getattr(block, "type", None)
        if block_type == "text":
            texts.append(getattr(block, "text", ""))

    return "\n".join(t for t in texts if t).strip()


def anthropic_usage_to_dict(message) -> dict:
    usage = getattr(message, "usage", None)
    if usage is None:
        return {}

    try:
        return usage.model_dump()
    except Exception:
        try:
            return dict(usage)
        except Exception:
            return {}


def parse_anthropic_batch_results_to_standard_jsonl(
    batch_id: str,
    plan_csv_path: str | Path,
    scenario_name: str,
    provider_name: str,
    model_name: str,
    output_dir: Path = AI_DATA_DIR,
) -> Path:
    plan_df = pd.read_csv(plan_csv_path)

    plan_by_custom_id = {
        row["anthropic_custom_id"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    standard_output_path = output_dir / f"{provider_name}__{model_name}__{scenario_name}.jsonl"

    n_success = 0
    n_error = 0

    result_stream = anthropic_client.messages.batches.results(batch_id)

    raw_results_path = ANTHROPIC_BATCH_DIR / f"{batch_id}__results_raw.jsonl"

    for entry in result_stream:
        try:
            entry_dict = entry.model_dump()
        except Exception:
            entry_dict = {"repr": repr(entry)}

        append_jsonl(raw_results_path, entry_dict)

        custom_id = getattr(entry, "custom_id", None)
        plan_row = plan_by_custom_id.get(custom_id, {})

        result = getattr(entry, "result", None)
        result_type = getattr(result, "type", None)

        if result_type == "succeeded":
            message = getattr(result, "message", None)
            text = extract_text_from_anthropic_message(message)
            usage = anthropic_usage_to_dict(message)

            record = {
                **plan_row,
                "created_at_utc": now_iso(),
                "status": "success" if text else "empty_text",
                "text": text,
                "provider_response_id": getattr(message, "id", None),
                "usage": usage,
                "raw_response": None,
                "error": None if text else "No text extracted from Anthropic message.",
                "batch_custom_id": custom_id,
                "batch_id": batch_id,
                "batch_output_file": str(raw_results_path),
            }

            n_success += int(bool(text))
            n_error += int(not bool(text))

        else:
            error_obj = getattr(result, "error", None)
            try:
                error_dump = error_obj.model_dump() if error_obj is not None else None
            except Exception:
                error_dump = repr(error_obj)

            record = {
                **plan_row,
                "created_at_utc": now_iso(),
                "status": "error",
                "text": None,
                "provider_response_id": None,
                "usage": None,
                "raw_response": None,
                "error": error_dump,
                "batch_custom_id": custom_id,
                "batch_id": batch_id,
                "batch_output_file": str(raw_results_path),
            }

            n_error += 1

        append_jsonl(standard_output_path, record)

    print(f"Raw Anthropic results: {raw_results_path}")
    print(f"Parsed standard output: {standard_output_path}")
    print(f"Success-ish text records: {n_success:,}")
    print(f"Errors/empty:             {n_error:,}")

    return standard_output_path

In [135]:
main_batch_status = check_anthropic_batch(claude_main_batch["batch_id"])

if main_batch_status.processing_status != "ended":
    print("Batch not ended yet. Re-run this cell later.")
else:
    claude_main_standard_path = parse_anthropic_batch_results_to_standard_jsonl(
        batch_id=claude_main_batch["batch_id"],
        plan_csv_path=claude_main_batch["plan_csv_path"],
        scenario_name="neutral_main_t1",
        provider_name="anthropic",
        model_name=MODEL_NAME,
    )

    claude_main_df = pd.DataFrame(read_jsonl(claude_main_standard_path))
    print(claude_main_df.shape)
    print(claude_main_df["status"].value_counts(dropna=False))
    display(claude_main_df[["prompt_id", "temperature", "run_idx", "status", "text"]].head())

MessageBatch(id='msgbatch_01CVybKyD2VB9giNLvjkteid', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 4, 29, 2, 19, 1, 180704, tzinfo=datetime.timezone.utc), ended_at=datetime.datetime(2026, 4, 29, 2, 20, 13, 517323, tzinfo=TzInfo(UTC)), expires_at=datetime.datetime(2026, 4, 30, 2, 19, 1, 180704, tzinfo=datetime.timezone.utc), processing_status='ended', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=150), results_url='https://api.anthropic.com/v1/messages/batches/msgbatch_01CVybKyD2VB9giNLvjkteid/results', type='message_batch')
Raw Anthropic results: ai_data/anthropic_batches/msgbatch_01CVybKyD2VB9giNLvjkteid__results_raw.jsonl
Parsed standard output: ai_data/story_generations/anthropic__claude-sonnet-4-5__neutral_main_t1.jsonl
Success-ish text records: 150
Errors/empty:             0
(150, 25)
status
success    150
Name: count, dtype: int64


,prompt_id,temperature,run_idx,status,text
0,10491,1.0,0,success,**The Last Visitor**\n\nThe nursing home hallw...
1,10491,1.0,1,success,"**The Nursery Monitor**\n\nAt 3:17 AM, the bab..."
2,10491,1.0,2,success,**The Babysitter**\n\nThe children were finall...
3,10491,1.0,3,success,**The Smile**\n\nI found my daughter standing ...
4,10491,1.0,4,success,**The Babysitter**\n\nThe children were finall...


In [136]:
claude_main_df["word_count"] = claude_main_df["text"].fillna("").str.findall(r"\b[\w']+\b").str.len()

claude_main_df.groupby("prompt_id").agg(
    n=("text", "size"),
    mean_words=("word_count", "mean"),
    median_words=("word_count", "median"),
    min_words=("word_count", "min"),
    max_words=("word_count", "max"),
)

,n,mean_words,median_words,min_words,max_words
prompt_id,,,,,
10491,50,101.14,101.0,94,107
93742,50,100.72,101.0,89,109
93855,50,113.20,114.0,104,123


In [137]:
claude_temp_batch = submit_anthropic_batch(
    scenario_name="neutral_temperature_robustness",
    model_name=MODEL_NAME,
    prompts_df=selected_story_prompts,
    exclude_existing_successes=True,
)

claude_personality_batch = submit_anthropic_batch(
    scenario_name="personality_grid",
    model_name=MODEL_NAME,
    prompts_df=selected_story_prompts,
    exclude_existing_successes=True,
)

claude_robustness_batches = pd.DataFrame([claude_temp_batch, claude_personality_batch])
claude_robustness_batches

Scenario: neutral_temperature_robustness
Requests to submit: 60
Plan CSV: ai_data/anthropic_batches/anthropic__claude-sonnet-4-5__neutral_temperature_robustness__20260428_222245__batch_plan.csv
Request JSON: ai_data/anthropic_batches/anthropic__claude-sonnet-4-5__neutral_temperature_robustness__20260428_222245__batch_requests.json
Submitted Anthropic batch:
{
  "batch_id": "msgbatch_013Dq79Cz9FmqFhxLTm5iZfa",
  "scenario_name": "neutral_temperature_robustness",
  "model": "claude-sonnet-4-5",
  "submitted_at_utc": "2026-04-29T02:22:45.656720+00:00",
  "n_requests_submitted": 60,
  "plan_csv_path": "ai_data/anthropic_batches/anthropic__claude-sonnet-4-5__neutral_temperature_robustness__20260428_222245__batch_plan.csv",
  "request_json_path": "ai_data/anthropic_batches/anthropic__claude-sonnet-4-5__neutral_temperature_robustness__20260428_222245__batch_requests.json",
  "processing_status": "in_progress"
}
Scenario: personality_grid
Requests to submit: 2,880
Plan CSV: ai_data/anthropic_b

,batch_id,scenario_name,model,submitted_at_utc,n_requests_submitted,plan_csv_path,request_json_path,processing_status
0,msgbatch_013Dq79Cz9FmqFhxLTm5iZfa,neutral_temperature_robustness,claude-sonnet-4-5,2026-04-29T02:22:45.656720+00:00,60,ai_data/anthropic_batches/anthropic__claude-so...,ai_data/anthropic_batches/anthropic__claude-so...,in_progress
1,msgbatch_01UgwEKen2Aa5rWXMZapkJqb,personality_grid,claude-sonnet-4-5,2026-04-29T02:22:46.786470+00:00,2880,ai_data/anthropic_batches/anthropic__claude-so...,ai_data/anthropic_batches/anthropic__claude-so...,in_progress


In [138]:
manifest_path = ANTHROPIC_BATCH_DIR / f"anthropic__{MODEL_NAME}__robustness_batch_manifest.csv"
claude_robustness_batches.to_csv(manifest_path, index=False)
manifest_path

PosixPath('ai_data/anthropic_batches/anthropic__claude-sonnet-4-5__robustness_batch_manifest.csv')

In [142]:
def check_anthropic_batches(batch_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in batch_df.iterrows():
        if pd.isna(row["batch_id"]) or row["batch_id"] is None:
            continue

        batch = anthropic_client.messages.batches.retrieve(row["batch_id"])

        request_counts = getattr(batch, "request_counts", None)

        rows.append({
            "batch_id": batch.id,
            "scenario_name": row["scenario_name"],
            "model": row["model"],
            "processing_status": getattr(batch, "processing_status", None),
            "ended_at": str(getattr(batch, "ended_at", None)),
            "created_at": str(getattr(batch, "created_at", None)),
            "expires_at": str(getattr(batch, "expires_at", None)),
            "request_counts": str(request_counts),
            "plan_csv_path": row["plan_csv_path"],
        })

    return pd.DataFrame(rows)


check_anthropic_batches(claude_robustness_batches)

,batch_id,scenario_name,model,processing_status,ended_at,created_at,expires_at,request_counts,plan_csv_path
0,msgbatch_013Dq79Cz9FmqFhxLTm5iZfa,neutral_temperature_robustness,claude-sonnet-4-5,ended,2026-04-29 02:25:28.261054+00:00,2026-04-29 02:22:45.488771+00:00,2026-04-30 02:22:45.488771+00:00,"MessageBatchRequestCounts(canceled=0, errored=...",ai_data/anthropic_batches/anthropic__claude-so...
1,msgbatch_01UgwEKen2Aa5rWXMZapkJqb,personality_grid,claude-sonnet-4-5,ended,2026-04-29 02:29:45.256450+00:00,2026-04-29 02:22:46.561299+00:00,2026-04-30 02:22:46.561299+00:00,"MessageBatchRequestCounts(canceled=0, errored=...",ai_data/anthropic_batches/anthropic__claude-so...


In [143]:
parsed_paths = []

for _, row in claude_robustness_batches.iterrows():
    batch = anthropic_client.messages.batches.retrieve(row["batch_id"])

    if batch.processing_status != "ended":
        print(f"Skipping {row['scenario_name']} because status is {batch.processing_status}")
        continue

    standard_path = parse_anthropic_batch_results_to_standard_jsonl(
        batch_id=row["batch_id"],
        plan_csv_path=row["plan_csv_path"],
        scenario_name=row["scenario_name"],
        provider_name="anthropic",
        model_name=MODEL_NAME,
    )

    parsed_paths.append(standard_path)

parsed_paths

Raw Anthropic results: ai_data/anthropic_batches/msgbatch_013Dq79Cz9FmqFhxLTm5iZfa__results_raw.jsonl
Parsed standard output: ai_data/story_generations/anthropic__claude-sonnet-4-5__neutral_temperature_robustness.jsonl
Success-ish text records: 30
Errors/empty:             30
Raw Anthropic results: ai_data/anthropic_batches/msgbatch_01UgwEKen2Aa5rWXMZapkJqb__results_raw.jsonl
Parsed standard output: ai_data/story_generations/anthropic__claude-sonnet-4-5__personality_grid.jsonl
Success-ish text records: 1,920
Errors/empty:             960


[PosixPath('ai_data/story_generations/anthropic__claude-sonnet-4-5__neutral_temperature_robustness.jsonl'),
 PosixPath('ai_data/story_generations/anthropic__claude-sonnet-4-5__personality_grid.jsonl')]

In [144]:
SCENARIOS["anthropic_neutral_temperature_robustness"] = GenerationScenario(
    scenario_name="anthropic_neutral_temperature_robustness",
    condition_type="neutral",
    temperatures=[0.3],  # T=0.7 already collected; T=1.0 is main benchmark
    n_per_prompt=10,
    prompt_ids=SELECTED_PROMPT_IDS,
    include_personas=False,
    max_output_tokens=800,
    description="Anthropic temperature robustness: add T=0.3; T=0.7 was already collected, T=1.0 is main benchmark.",
)

SCENARIOS["anthropic_personality_grid_t03"] = GenerationScenario(
    scenario_name="anthropic_personality_grid_t03",
    condition_type="personality",
    temperatures=[0.3],
    n_per_prompt=10,
    prompt_ids=SELECTED_PROMPT_IDS,
    include_personas=True,
    max_output_tokens=800,
    description="Anthropic personality robustness backfill at T=0.3, replacing invalid T=1.3.",
)

In [145]:
claude_temp_t03_batch = submit_anthropic_batch(
    scenario_name="anthropic_neutral_temperature_robustness",
    model_name=MODEL_NAME,
    prompts_df=selected_story_prompts,
    exclude_existing_successes=True,
)

claude_personality_t03_batch = submit_anthropic_batch(
    scenario_name="anthropic_personality_grid_t03",
    model_name=MODEL_NAME,
    prompts_df=selected_story_prompts,
    exclude_existing_successes=True,
)

claude_t03_batches = pd.DataFrame([claude_temp_t03_batch, claude_personality_t03_batch])
claude_t03_batches

Scenario: anthropic_neutral_temperature_robustness
Requests to submit: 30
Plan CSV: ai_data/anthropic_batches/anthropic__claude-sonnet-4-5__anthropic_neutral_temperature_robustness__20260428_223325__batch_plan.csv
Request JSON: ai_data/anthropic_batches/anthropic__claude-sonnet-4-5__anthropic_neutral_temperature_robustness__20260428_223325__batch_requests.json
Submitted Anthropic batch:
{
  "batch_id": "msgbatch_01CA2VJCeH3ARwMj5Jwu8m7i",
  "scenario_name": "anthropic_neutral_temperature_robustness",
  "model": "claude-sonnet-4-5",
  "submitted_at_utc": "2026-04-29T02:33:25.847586+00:00",
  "n_requests_submitted": 30,
  "plan_csv_path": "ai_data/anthropic_batches/anthropic__claude-sonnet-4-5__anthropic_neutral_temperature_robustness__20260428_223325__batch_plan.csv",
  "request_json_path": "ai_data/anthropic_batches/anthropic__claude-sonnet-4-5__anthropic_neutral_temperature_robustness__20260428_223325__batch_requests.json",
  "processing_status": "in_progress"
}
Scenario: anthropic_pe

,batch_id,scenario_name,model,submitted_at_utc,n_requests_submitted,plan_csv_path,request_json_path,processing_status
0,msgbatch_01CA2VJCeH3ARwMj5Jwu8m7i,anthropic_neutral_temperature_robustness,claude-sonnet-4-5,2026-04-29T02:33:25.847586+00:00,30,ai_data/anthropic_batches/anthropic__claude-so...,ai_data/anthropic_batches/anthropic__claude-so...,in_progress
1,msgbatch_01JzUvjWcHQyUY2Qb8B9F4nh,anthropic_personality_grid_t03,claude-sonnet-4-5,2026-04-29T02:33:26.500502+00:00,960,ai_data/anthropic_batches/anthropic__claude-so...,ai_data/anthropic_batches/anthropic__claude-so...,in_progress


In [148]:
check_anthropic_batches(claude_t03_batches)

,batch_id,scenario_name,model,processing_status,ended_at,created_at,expires_at,request_counts,plan_csv_path
0,msgbatch_01CA2VJCeH3ARwMj5Jwu8m7i,anthropic_neutral_temperature_robustness,claude-sonnet-4-5,ended,2026-04-29 02:35:52.412567+00:00,2026-04-29 02:33:25.751185+00:00,2026-04-30 02:33:25.751185+00:00,"MessageBatchRequestCounts(canceled=0, errored=...",ai_data/anthropic_batches/anthropic__claude-so...
1,msgbatch_01JzUvjWcHQyUY2Qb8B9F4nh,anthropic_personality_grid_t03,claude-sonnet-4-5,ended,2026-04-29 02:36:23.981587+00:00,2026-04-29 02:33:26.353928+00:00,2026-04-30 02:33:26.353928+00:00,"MessageBatchRequestCounts(canceled=0, errored=...",ai_data/anthropic_batches/anthropic__claude-so...


In [149]:
parsed_paths = []

for _, row in claude_t03_batches.iterrows():
    batch = anthropic_client.messages.batches.retrieve(row["batch_id"])

    if batch.processing_status != "ended":
        print(f"Skipping {row['scenario_name']} because status is {batch.processing_status}")
        continue

    standard_path = parse_anthropic_batch_results_to_standard_jsonl(
        batch_id=row["batch_id"],
        plan_csv_path=row["plan_csv_path"],
        scenario_name=row["scenario_name"],
        provider_name="anthropic",
        model_name=MODEL_NAME,
    )

    parsed_paths.append(standard_path)

parsed_paths

Raw Anthropic results: ai_data/anthropic_batches/msgbatch_01CA2VJCeH3ARwMj5Jwu8m7i__results_raw.jsonl
Parsed standard output: ai_data/story_generations/anthropic__claude-sonnet-4-5__anthropic_neutral_temperature_robustness.jsonl
Success-ish text records: 30
Errors/empty:             0
Raw Anthropic results: ai_data/anthropic_batches/msgbatch_01JzUvjWcHQyUY2Qb8B9F4nh__results_raw.jsonl
Parsed standard output: ai_data/story_generations/anthropic__claude-sonnet-4-5__anthropic_personality_grid_t03.jsonl
Success-ish text records: 960
Errors/empty:             0


[PosixPath('ai_data/story_generations/anthropic__claude-sonnet-4-5__anthropic_neutral_temperature_robustness.jsonl'),
 PosixPath('ai_data/story_generations/anthropic__claude-sonnet-4-5__anthropic_personality_grid_t03.jsonl')]

In [150]:
all_claude_story_df = load_all_generation_files(
    provider_name="anthropic",
    model_name=MODEL_NAME,
    output_dir=AI_DATA_DIR,
)

all_claude_story_df_dedup = deduplicate_generation_records(all_claude_story_df)

all_claude_story_df_dedup["analysis_scenario_name"] = all_claude_story_df_dedup["scenario_name"]

all_claude_story_df_dedup.loc[
    all_claude_story_df_dedup["scenario_name"].isin([
        "neutral_temperature_robustness",
        "anthropic_neutral_temperature_robustness",
    ]),
    "analysis_scenario_name"
] = "neutral_temperature_robustness"

all_claude_story_df_dedup.loc[
    all_claude_story_df_dedup["scenario_name"].isin([
        "personality_grid",
        "anthropic_personality_grid_t03",
    ]),
    "analysis_scenario_name"
] = "personality_grid"

In [151]:
all_claude_story_df_dedup.groupby(
    ["analysis_scenario_name", "temperature"], dropna=False
).size()

analysis_scenario_name          temperature
neutral_main_t1                 1.0            150
neutral_temperature_robustness  0.3             30
                                0.7             30
                                1.3             30
personality_grid                0.3            960
                                0.7            960
                                1.0            960
                                1.3            960
dtype: int64

In [152]:
all_claude_story_df_dedup.groupby(
    ["analysis_scenario_name", "temperature", "status"], dropna=False
).size()

analysis_scenario_name          temperature  status 
neutral_main_t1                 1.0          success    150
neutral_temperature_robustness  0.3          success     30
                                0.7          success     30
                                1.3          error       30
personality_grid                0.3          success    960
                                0.7          success    960
                                1.0          success    960
                                1.3          error      960
dtype: int64

In [153]:
claude_success_df = all_claude_story_df_dedup.query("status == 'success'").copy()

claude_success_df.groupby(
    ["analysis_scenario_name", "temperature"], dropna=False
).size()

analysis_scenario_name          temperature
neutral_main_t1                 1.0            150
neutral_temperature_robustness  0.3             30
                                0.7             30
personality_grid                0.3            960
                                0.7            960
                                1.0            960
dtype: int64

In [154]:
CLEAN_AI_DIR = Path("ai_data/processed")
CLEAN_AI_DIR.mkdir(parents=True, exist_ok=True)

claude_success_path_pkl = CLEAN_AI_DIR / f"anthropic__{MODEL_NAME}__story_generations_success_only.pkl"
claude_success_path_csv = CLEAN_AI_DIR / f"anthropic__{MODEL_NAME}__story_generations_success_only.csv"

claude_success_df.to_pickle(claude_success_path_pkl)
claude_success_df.to_csv(claude_success_path_csv, index=False)

print(claude_success_path_pkl)
print(claude_success_path_csv)

ai_data/processed/anthropic__claude-sonnet-4-5__story_generations_success_only.pkl
ai_data/processed/anthropic__claude-sonnet-4-5__story_generations_success_only.csv


# gemini

In [155]:
!pip install -U google-genai pandas tqdm python-dotenv

In [164]:
import os

os.environ["GEMINI_API_KEY"] = ""

In [165]:
from pathlib import Path
import json
import time
import datetime as dt
import pandas as pd

from google import genai
from google.genai import types

gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

PROVIDER_NAME = "gemini"
MODEL_NAME = "gemini-2.5-pro"

GEMINI_BATCH_DIR = Path("ai_data/gemini_batches")
GEMINI_BATCH_DIR.mkdir(parents=True, exist_ok=True)

AI_DATA_DIR = Path("ai_data/story_generations")
AI_DATA_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_AI_DIR = Path("ai_data/processed")
CLEAN_AI_DIR.mkdir(parents=True, exist_ok=True)

In [166]:
SCENARIOS["gemini_neutral_temperature_robustness"] = GenerationScenario(
    scenario_name="gemini_neutral_temperature_robustness",
    condition_type="neutral",
    temperatures=[0.7, 1.3],  # T=1.0 covered by neutral_main_t1
    n_per_prompt=10,
    prompt_ids=SELECTED_PROMPT_IDS,
    include_personas=False,
    max_output_tokens=800,
    description="Gemini temperature robustness: neutral prompt at T=0.7 and T=1.3.",
)

SCENARIOS["gemini_personality_grid"] = GenerationScenario(
    scenario_name="gemini_personality_grid",
    condition_type="personality",
    temperatures=[0.7, 1.0, 1.3],
    n_per_prompt=10,
    prompt_ids=SELECTED_PROMPT_IDS,
    include_personas=True,
    max_output_tokens=800,
    description="Gemini personality robustness: 32 Big-Five-style profiles crossed with temperature.",
)

In [167]:
test_response = gemini_client.models.generate_content(
    model=MODEL_NAME,
    contents=[
        types.Content(
            role="user",
            parts=[types.Part(text=build_user_prompt(selected_story_prompts.loc[0, "prompt"]))]
        )
    ],
    config=types.GenerateContentConfig(
        system_instruction=NEUTRAL_SYSTEM_INSTRUCTIONS,
        temperature=1.3,
        max_output_tokens=800,
    ),
)

print(test_response.text)

My daughter won’t stop crying and pointing at the basement door. I try to soothe her, telling her there’s nothing down there but old furniture and spiders. She just wails louder, her little body trembling. “He’s mad,” she sobs, “You locked him in.” To prove she’s wrong, I unlock the door and swing it open, revealing the dark, empty staircase. Her crying stops instantly. A tiny, raspy voice from the darkness whispers, “Thank you.” My daughter smiles, takes the unseen hand offered from the shadows, and skips down the stairs. The door slams shut.


In [168]:
def make_gemini_custom_id(request_key: str) -> str:
    return "r_" + request_key[:48]


def build_gemini_generation_plan(
    scenario_name: str,
    model_name: str,
    prompts_df: pd.DataFrame,
) -> pd.DataFrame:
    scenario = SCENARIOS[scenario_name]

    plan_df = build_generation_plan(
        scenario=scenario,
        prompts_df=prompts_df,
        provider_name="gemini",
        model_name=model_name,
    ).copy()

    plan_df["gemini_custom_id"] = plan_df["request_key"].map(make_gemini_custom_id)

    if plan_df["gemini_custom_id"].duplicated().any():
        raise ValueError("Duplicate gemini_custom_id found. Increase hash length.")

    return plan_df

In [169]:
def make_gemini_inline_requests(plan_df: pd.DataFrame) -> list:
    inline_requests = []

    for _, row in plan_df.iterrows():
        req = {
            "key": row["gemini_custom_id"],
            "request": {
                "contents": [
                    {
                        "role": "user",
                        "parts": [
                            {"text": row["user_prompt"]}
                        ],
                    }
                ],
                "config": {
                    "system_instruction": {
                        "parts": [
                            {"text": row["system_instructions"]}
                        ]
                    },
                    "temperature": float(row["temperature"]),
                    "max_output_tokens": int(row["max_output_tokens"]),
                },
            },
        }
        inline_requests.append(req)

    return inline_requests

In [172]:
def make_gemini_batch_jsonl_file(
    scenario_name: str,
    model_name: str,
    prompts_df: pd.DataFrame,
    output_dir: Path = GEMINI_BATCH_DIR,
    exclude_existing_successes: bool = True,
) -> tuple[Path, Path, pd.DataFrame]:
    """
    Create a Gemini batch JSONL file.

    File format:
    {"key": "...", "request": {"contents": [...], "generation_config": {...}, "system_instruction": {...}}}

    This key/request format is appropriate for file-based Gemini batch input.
    """
    standard_output_path = AI_DATA_DIR / f"gemini__{model_name}__{scenario_name}.jsonl"

    plan_df = build_gemini_generation_plan(
        scenario_name=scenario_name,
        model_name=model_name,
        prompts_df=prompts_df,
    )

    if exclude_existing_successes and standard_output_path.exists():
        done_keys = successful_request_keys(standard_output_path)
        plan_df = plan_df[~plan_df["request_key"].isin(done_keys)].copy()

    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"gemini__{model_name}__{scenario_name}__{timestamp}"

    plan_csv_path = output_dir / f"{stem}__batch_plan.csv"
    batch_jsonl_path = output_dir / f"{stem}__batch_input.jsonl"

    plan_df.to_csv(plan_csv_path, index=False)

    with open(batch_jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            request_obj = {
                "key": row["gemini_custom_id"],
                "request": {
                    "contents": [
                        {
                            "role": "user",
                            "parts": [
                                {"text": row["user_prompt"]}
                            ],
                        }
                    ],
                    # Gemini batch file examples often use generation_config.
                    # This is the REST-style field name.
                    "generation_config": {
                        "temperature": float(row["temperature"]),
                        "max_output_tokens": int(row["max_output_tokens"]),
                        "response_modalities": ["TEXT"],
                    },
                    "system_instruction": {
                        "parts": [
                            {"text": row["system_instructions"]}
                        ]
                    },
                },
            }

            f.write(json.dumps(request_obj, ensure_ascii=False) + "\n")

    print(f"Scenario: {scenario_name}")
    print(f"Requests written: {len(plan_df):,}")
    print(f"Plan CSV:     {plan_csv_path}")
    print(f"Batch JSONL:  {batch_jsonl_path}")

    return batch_jsonl_path, plan_csv_path, plan_df

In [170]:
# def submit_gemini_batch(
#     scenario_name: str,
#     model_name: str,
#     prompts_df: pd.DataFrame,
#     output_dir: Path = GEMINI_BATCH_DIR,
#     exclude_existing_successes: bool = True,
# ) -> dict:
#     standard_output_path = AI_DATA_DIR / f"gemini__{model_name}__{scenario_name}.jsonl"

#     plan_df = build_gemini_generation_plan(
#         scenario_name=scenario_name,
#         model_name=model_name,
#         prompts_df=prompts_df,
#     )

#     if exclude_existing_successes and standard_output_path.exists():
#         done_keys = successful_request_keys(standard_output_path)
#         plan_df = plan_df[~plan_df["request_key"].isin(done_keys)].copy()

#     timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
#     stem = f"gemini__{model_name}__{scenario_name}__{timestamp}"

#     plan_csv_path = output_dir / f"{stem}__batch_plan.csv"
#     requests_json_path = output_dir / f"{stem}__inline_requests.json"

#     plan_df.to_csv(plan_csv_path, index=False)

#     inline_requests = make_gemini_inline_requests(plan_df)

#     with open(requests_json_path, "w", encoding="utf-8") as f:
#         json.dump(inline_requests, f, ensure_ascii=False, indent=2)

#     print(f"Scenario: {scenario_name}")
#     print(f"Requests to submit: {len(inline_requests):,}")
#     print(f"Plan CSV: {plan_csv_path}")
#     print(f"Request JSON: {requests_json_path}")

#     if len(inline_requests) == 0:
#         return {
#             "batch_name": None,
#             "scenario_name": scenario_name,
#             "model": model_name,
#             "submitted_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
#             "n_requests_submitted": 0,
#             "plan_csv_path": str(plan_csv_path),
#             "requests_json_path": str(requests_json_path),
#         }

#     batch_job = gemini_client.batches.create(
#         model=model_name,
#         src={"inlined_requests": inline_requests},
#         config={"display_name": stem},
#     )

#     batch_info = {
#         "batch_name": batch_job.name,
#         "scenario_name": scenario_name,
#         "model": model_name,
#         "submitted_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
#         "n_requests_submitted": len(inline_requests),
#         "plan_csv_path": str(plan_csv_path),
#         "requests_json_path": str(requests_json_path),
#         "raw_state": str(getattr(batch_job, "state", None)),
#     }

#     manifest_path = output_dir / f"{stem}__batch_manifest.json"
#     with open(manifest_path, "w", encoding="utf-8") as f:
#         json.dump(batch_info, f, indent=2)

#     print("Submitted Gemini batch:")
#     print(json.dumps(batch_info, indent=2))

#     return batch_info

In [175]:
def submit_gemini_batch_from_jsonl(
    scenario_name: str,
    model_name: str,
    prompts_df: pd.DataFrame,
    output_dir: Path = GEMINI_BATCH_DIR,
    exclude_existing_successes: bool = True,
) -> dict:
    batch_jsonl_path, plan_csv_path, plan_df = make_gemini_batch_jsonl_file(
        scenario_name=scenario_name,
        model_name=model_name,
        prompts_df=prompts_df,
        output_dir=output_dir,
        exclude_existing_successes=exclude_existing_successes,
    )

    if len(plan_df) == 0:
        return {
            "batch_name": None,
            "scenario_name": scenario_name,
            "model": model_name,
            "submitted_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
            "n_requests_submitted": 0,
            "plan_csv_path": str(plan_csv_path),
            "batch_jsonl_path": str(batch_jsonl_path),
            "uploaded_file_name": None,
        }

    uploaded_file = gemini_client.files.upload(
        file=str(batch_jsonl_path),
        config={"mime_type": "application/jsonl"},
    )

    print(f"Uploaded file: {uploaded_file.name}")

    batch_job = gemini_client.batches.create(
        model=model_name,
        src=uploaded_file.name,
        config={"display_name": batch_jsonl_path.stem},
    )

    batch_info = {
        "batch_name": batch_job.name,
        "scenario_name": scenario_name,
        "model": model_name,
        "submitted_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "n_requests_submitted": len(plan_df),
        "plan_csv_path": str(plan_csv_path),
        "batch_jsonl_path": str(batch_jsonl_path),
        "uploaded_file_name": uploaded_file.name,
        "raw_state": str(getattr(batch_job, "state", None)),
    }

    manifest_path = output_dir / f"{batch_jsonl_path.stem}__batch_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(batch_info, f, indent=2)

    print("Submitted Gemini batch:")
    print(json.dumps(batch_info, indent=2))

    return batch_info

In [176]:
PROVIDER_NAME = "gemini"
MODEL_NAME = "gemini-2.5-pro"

gemini_main_batch = submit_gemini_batch_from_jsonl(
    scenario_name="neutral_main_t1",
    model_name=MODEL_NAME,
    prompts_df=selected_story_prompts,
    exclude_existing_successes=True,
)

gemini_main_batch

Scenario: neutral_main_t1
Requests written: 150
Plan CSV:     ai_data/gemini_batches/gemini__gemini-2.5-pro__neutral_main_t1__20260428_230353__batch_plan.csv
Batch JSONL:  ai_data/gemini_batches/gemini__gemini-2.5-pro__neutral_main_t1__20260428_230353__batch_input.jsonl
Uploaded file: files/sfqxib13ei9y
Submitted Gemini batch:
{
  "batch_name": "batches/06k90js6s0o5bwwjn36py055ab6btx4d9mb4",
  "scenario_name": "neutral_main_t1",
  "model": "gemini-2.5-pro",
  "submitted_at_utc": "2026-04-29T03:03:56.395191+00:00",
  "n_requests_submitted": 150,
  "plan_csv_path": "ai_data/gemini_batches/gemini__gemini-2.5-pro__neutral_main_t1__20260428_230353__batch_plan.csv",
  "batch_jsonl_path": "ai_data/gemini_batches/gemini__gemini-2.5-pro__neutral_main_t1__20260428_230353__batch_input.jsonl",
  "uploaded_file_name": "files/sfqxib13ei9y",
  "raw_state": "JobState.JOB_STATE_PENDING"
}


{'batch_name': 'batches/06k90js6s0o5bwwjn36py055ab6btx4d9mb4',
 'scenario_name': 'neutral_main_t1',
 'model': 'gemini-2.5-pro',
 'submitted_at_utc': '2026-04-29T03:03:56.395191+00:00',
 'n_requests_submitted': 150,
 'plan_csv_path': 'ai_data/gemini_batches/gemini__gemini-2.5-pro__neutral_main_t1__20260428_230353__batch_plan.csv',
 'batch_jsonl_path': 'ai_data/gemini_batches/gemini__gemini-2.5-pro__neutral_main_t1__20260428_230353__batch_input.jsonl',
 'uploaded_file_name': 'files/sfqxib13ei9y',
 'raw_state': 'JobState.JOB_STATE_PENDING'}

In [180]:
def check_gemini_batch(batch_name: str):
    batch = gemini_client.batches.get(name=batch_name)
    print(batch)
    return batch

In [182]:
gemini_main_status = check_gemini_batch(
    "batches/06k90js6s0o5bwwjn36py055ab6btx4d9mb4"
)

name='batches/06k90js6s0o5bwwjn36py055ab6btx4d9mb4' display_name='gemini__gemini-2.5-pro__neutral_main_t1__20260428_230353__batch_input' state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'> error=None create_time=datetime.datetime(2026, 4, 29, 3, 3, 55, 385509, tzinfo=TzInfo(UTC)) start_time=None end_time=datetime.datetime(2026, 4, 29, 3, 6, 55, 680202, tzinfo=TzInfo(UTC)) update_time=datetime.datetime(2026, 4, 29, 3, 6, 55, 680202, tzinfo=TzInfo(UTC)) model='models/gemini-2.5-pro' src=None dest=BatchJobDestination(
  file_name='files/batch-06k90js6s0o5bwwjn36py055ab6btx4d9mb4'
)


In [184]:
def gemini_obj_to_dict(obj):
    if obj is None:
        return None
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if hasattr(obj, "to_json_dict"):
        return obj.to_json_dict()
    try:
        return json.loads(obj.to_json())
    except Exception:
        return {"repr": repr(obj)}


def extract_text_from_gemini_response(response_obj) -> str:
    if response_obj is None:
        return ""

    # SDK convenience attribute
    text = getattr(response_obj, "text", None)
    if text:
        return str(text).strip()

    d = gemini_obj_to_dict(response_obj) or {}

    texts = []
    for cand in d.get("candidates", []) or []:
        content = cand.get("content", {}) or {}
        for part in content.get("parts", []) or []:
            if "text" in part:
                texts.append(part["text"])

    return "\n".join(texts).strip()


def gemini_usage_to_dict(response_obj) -> dict:
    if response_obj is None:
        return {}

    usage = getattr(response_obj, "usage_metadata", None)
    if usage is not None:
        return gemini_obj_to_dict(usage) or {}

    d = gemini_obj_to_dict(response_obj) or {}
    return d.get("usageMetadata") or d.get("usage_metadata") or {}


def parse_gemini_batch_results_to_standard_jsonl(
    batch_name: str,
    plan_csv_path: str | Path,
    scenario_name: str,
    provider_name: str,
    model_name: str,
    output_dir: Path = AI_DATA_DIR,
) -> Path:
    plan_df = pd.read_csv(plan_csv_path)
    plan_by_custom_id = {
        row["gemini_custom_id"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    standard_output_path = output_dir / f"{provider_name}__{model_name}__{scenario_name}.jsonl"
    raw_results_path = GEMINI_BATCH_DIR / f"{batch_name.replace('/', '_')}__results_raw.jsonl"

    # Depending on SDK version, this may be:
    #   client.batches.results(name=batch_name)
    # or results accessible from the completed job object.
    result_stream = gemini_client.batches.results(name=batch_name)

    n_success = 0
    n_error = 0

    for entry in result_stream:
        entry_dict = gemini_obj_to_dict(entry)
        append_jsonl(raw_results_path, entry_dict)

        custom_id = (
            entry_dict.get("key")
            or entry_dict.get("custom_id")
            or entry_dict.get("customId")
        )

        plan_row = plan_by_custom_id.get(custom_id, {})

        response_obj = getattr(entry, "response", None)
        error_obj = getattr(entry, "error", None)

        # Fallback through dict
        if response_obj is None and isinstance(entry_dict, dict):
            response_obj = entry_dict.get("response")
        if error_obj is None and isinstance(entry_dict, dict):
            error_obj = entry_dict.get("error")

        text = extract_text_from_gemini_response(response_obj)
        usage = gemini_usage_to_dict(response_obj)

        if text:
            record = {
                **plan_row,
                "created_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
                "status": "success",
                "text": text,
                "provider_response_id": None,
                "usage": usage,
                "raw_response": None,
                "error": None,
                "batch_custom_id": custom_id,
                "batch_name": batch_name,
                "batch_output_file": str(raw_results_path),
            }
            n_success += 1
        else:
            record = {
                **plan_row,
                "created_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
                "status": "error" if error_obj else "empty_text",
                "text": None,
                "provider_response_id": None,
                "usage": usage,
                "raw_response": None,
                "error": gemini_obj_to_dict(error_obj),
                "batch_custom_id": custom_id,
                "batch_name": batch_name,
                "batch_output_file": str(raw_results_path),
            }
            n_error += 1

        append_jsonl(standard_output_path, record)

    print(f"Raw Gemini results: {raw_results_path}")
    print(f"Parsed standard output: {standard_output_path}")
    print(f"Success-ish text records: {n_success:,}")
    print(f"Errors/empty:             {n_error:,}")

    return standard_output_path

In [185]:
gemini_main_standard_path = parse_gemini_batch_results_to_standard_jsonl(
    batch_name="batches/06k90js6s0o5bwwjn36py055ab6btx4d9mb4",
    plan_csv_path=Path("ai_data/gemini_batches/gemini__gemini-2.5-pro__neutral_main_t1__20260428_230353__batch_plan.csv"),
    scenario_name="neutral_main_t1",
    provider_name="gemini",
    model_name="gemini-2.5-pro",
)

gemini_main_df = pd.DataFrame(read_jsonl(gemini_main_standard_path))

print(gemini_main_df.shape)
print(gemini_main_df["status"].value_counts(dropna=False))
gemini_main_df[["prompt_id", "temperature", "run_idx", "status", "text"]].head()

AttributeError: 'Batches' object has no attribute 'results'

In [186]:
job = gemini_client.batches.get(
    name="batches/06k90js6s0o5bwwjn36py055ab6btx4d9mb4"
)

print("JOB OBJECT:")
print(job)

print("\nATTRIBUTES:")
print([a for a in dir(job) if not a.startswith("_")])

print("\nSTATE:")
print(getattr(job, "state", None))

print("\nDEST:")
print(getattr(job, "dest", None))

print("\nOUTPUT:")
print(getattr(job, "output", None))

JOB OBJECT:
name='batches/06k90js6s0o5bwwjn36py055ab6btx4d9mb4' display_name='gemini__gemini-2.5-pro__neutral_main_t1__20260428_230353__batch_input' state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'> error=None create_time=datetime.datetime(2026, 4, 29, 3, 3, 55, 385509, tzinfo=TzInfo(UTC)) start_time=None end_time=datetime.datetime(2026, 4, 29, 3, 6, 55, 680202, tzinfo=TzInfo(UTC)) update_time=datetime.datetime(2026, 4, 29, 3, 6, 55, 680202, tzinfo=TzInfo(UTC)) model='models/gemini-2.5-pro' src=None dest=BatchJobDestination(
  file_name='files/batch-06k90js6s0o5bwwjn36py055ab6btx4d9mb4'
)

ATTRIBUTES:
['construct', 'copy', 'create_time', 'dest', 'dict', 'display_name', 'done', 'end_time', 'error', 'from_orm', 'json', 'model', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy', 'model_dump', 'model_dump_json', 'model_extra', 'model_fields', 'model_fields_set', 'model_json_schema', 'model_parametrized_name', 'model_post_init', 'model_rebuild', 'model_valid

In [187]:
def download_gemini_batch_output_file(
    batch_name: str,
    output_dir: Path = GEMINI_BATCH_DIR,
) -> Path:
    """
    Download the output file produced by a Gemini batch job.

    For file-based Gemini Developer API batches, the completed job has:
        job.dest.file_name = 'files/...'
    """
    job = gemini_client.batches.get(name=batch_name)

    print("Batch state:", getattr(job, "state", None))
    print("Batch error:", getattr(job, "error", None))
    print("Batch dest:", getattr(job, "dest", None))

    dest = getattr(job, "dest", None)
    file_name = getattr(dest, "file_name", None)

    if not file_name:
        raise ValueError(f"No dest.file_name found on batch job: {batch_name}")

    safe_name = batch_name.replace("/", "_")
    output_path = output_dir / f"{safe_name}__output.jsonl"

    # Current google-genai SDK supports files.download(file=..., download_path=...)
    # in recent versions. If this errors, see fallback cell below.
    gemini_client.files.download(
        file=file_name,
        download_path=str(output_path),
    )

    print(f"Downloaded Gemini batch output to: {output_path}")
    return output_path

In [188]:
gemini_main_output_path = download_gemini_batch_output_file(
    "batches/06k90js6s0o5bwwjn36py055ab6btx4d9mb4",
    output_dir=GEMINI_BATCH_DIR,
)

gemini_main_output_path

Batch state: JobState.JOB_STATE_SUCCEEDED
Batch error: None
Batch dest: format=None gcs_uri=None bigquery_uri=None file_name='files/batch-06k90js6s0o5bwwjn36py055ab6btx4d9mb4' inlined_responses=None inlined_embed_content_responses=None


TypeError: download() got an unexpected keyword argument 'download_path'

In [189]:
import inspect
print(inspect.signature(gemini_client.files.download))

(*, file: Union[str, google.genai.types.File, google.genai.types.Video, google.genai.types.GeneratedVideo], config: Union[google.genai.types.DownloadFileConfig, google.genai.types.DownloadFileConfigDict, NoneType] = None) -> bytes


In [190]:
def download_gemini_batch_output_file(
    batch_name: str,
    output_dir: Path = GEMINI_BATCH_DIR,
) -> Path:
    """
    Download the output file produced by a Gemini batch job.

    This version matches your installed google-genai SDK:
        files.download(file=...) -> bytes
    """
    job = gemini_client.batches.get(name=batch_name)

    print("Batch state:", getattr(job, "state", None))
    print("Batch error:", getattr(job, "error", None))
    print("Batch dest:", getattr(job, "dest", None))

    dest = getattr(job, "dest", None)
    file_name = getattr(dest, "file_name", None)

    if not file_name:
        raise ValueError(f"No dest.file_name found on batch job: {batch_name}")

    safe_name = batch_name.replace("/", "_")
    output_path = output_dir / f"{safe_name}__output.jsonl"

    downloaded_bytes = gemini_client.files.download(file=file_name)

    if not isinstance(downloaded_bytes, (bytes, bytearray)):
        raise TypeError(f"Expected bytes from files.download, got {type(downloaded_bytes)}")

    output_path.write_bytes(downloaded_bytes)

    print(f"Downloaded Gemini batch output to: {output_path}")
    print(f"File size: {output_path.stat().st_size:,} bytes")

    return output_path

In [191]:
gemini_main_output_path = download_gemini_batch_output_file(
    "batches/06k90js6s0o5bwwjn36py055ab6btx4d9mb4",
    output_dir=GEMINI_BATCH_DIR,
)

gemini_main_output_path

Batch state: JobState.JOB_STATE_SUCCEEDED
Batch error: None
Batch dest: format=None gcs_uri=None bigquery_uri=None file_name='files/batch-06k90js6s0o5bwwjn36py055ab6btx4d9mb4' inlined_responses=None inlined_embed_content_responses=None
Downloaded Gemini batch output to: ai_data/gemini_batches/batches_06k90js6s0o5bwwjn36py055ab6btx4d9mb4__output.jsonl
File size: 69,887 bytes


PosixPath('ai_data/gemini_batches/batches_06k90js6s0o5bwwjn36py055ab6btx4d9mb4__output.jsonl')

In [192]:
gemini_raw_records = read_jsonl(gemini_main_output_path)

print(f"Raw records: {len(gemini_raw_records):,}")

for r in gemini_raw_records[:3]:
    print("=" * 100)
    print(json.dumps(r, indent=2)[:4000])

Raw records: 150
{
  "response": {
    "modelVersion": "gemini-2.5-pro",
    "responseId": "YHXxaZ68AuLRz7IPxJm66A0",
    "candidates": [
      {
        "content": {
          "role": "model"
        },
        "index": 0,
        "finishReason": "MAX_TOKENS"
      }
    ],
    "usageMetadata": {
      "totalTokenCount": 862,
      "thoughtsTokenCount": 797,
      "promptTokensDetails": [
        {
          "tokenCount": 65,
          "modality": "TEXT"
        }
      ],
      "promptTokenCount": 65
    }
  },
  "key": "r_407aae8ae53fffd3bb3cdfc289008a2116cfa651d53ab1c7"
}
{
  "response": {
    "responseId": "YHXxae-zKP2Fz7IPs5DjwAo",
    "candidates": [
      {
        "index": 0,
        "finishReason": "MAX_TOKENS",
        "content": {
          "role": "model"
        }
      }
    ],
    "usageMetadata": {
      "thoughtsTokenCount": 797,
      "promptTokensDetails": [
        {
          "tokenCount": 65,
          "modality": "TEXT"
        }
      ],
      "promptTokenCount

In [194]:
def extract_text_from_gemini_response_dict(response: dict) -> str:
    if not isinstance(response, dict):
        return ""

    # Sometimes response text may be flattened.
    if response.get("text"):
        return str(response["text"]).strip()

    texts = []

    for cand in response.get("candidates", []) or []:
        content = cand.get("content", {}) or {}
        for part in content.get("parts", []) or []:
            if isinstance(part, dict) and part.get("text"):
                texts.append(part["text"])

    return "\n".join(texts).strip()


def extract_gemini_usage_dict(response: dict) -> dict:
    if not isinstance(response, dict):
        return {}

    return (
        response.get("usageMetadata")
        or response.get("usage_metadata")
        or response.get("usage")
        or {}
    )


def parse_gemini_downloaded_batch_jsonl_to_standard_jsonl(
    downloaded_output_path: Path,
    plan_csv_path: str | Path,
    scenario_name: str,
    provider_name: str,
    model_name: str,
    output_dir: Path = AI_DATA_DIR,
) -> Path:
    plan_df = pd.read_csv(plan_csv_path)
    plan_by_custom_id = {
        row["gemini_custom_id"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    raw_records = read_jsonl(downloaded_output_path)

    standard_output_path = output_dir / f"{provider_name}__{model_name}__{scenario_name}.jsonl"

    n_success = 0
    n_error = 0

    for rec in raw_records:
        custom_id = (
            rec.get("key")
            or rec.get("custom_id")
            or rec.get("customId")
        )

        plan_row = plan_by_custom_id.get(custom_id, {})

        # Common shapes:
        # 1. {"key": "...", "response": {...}}
        # 2. {"key": "...", "response": {"candidates": ...}}
        # 3. {"key": "...", "error": {...}}
        response = rec.get("response") or rec.get("result") or {}
        error = rec.get("error")

        # Some variants wrap response inside response.body
        if isinstance(response, dict) and "body" in response:
            response = response.get("body") or {}

        text = extract_text_from_gemini_response_dict(response)
        usage = extract_gemini_usage_dict(response)

        if text:
            status = "success"
            n_success += 1
            error_out = None
        else:
            status = "error" if error else "empty_text"
            n_error += 1
            error_out = error or "No text extracted from Gemini response."

        standard_record = {
            **plan_row,
            "created_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
            "status": status,
            "text": text if text else None,
            "provider_response_id": None,
            "usage": usage,
            "raw_response": None,
            "error": error_out,
            "batch_custom_id": custom_id,
            "batch_name": None,
            "batch_output_file": str(downloaded_output_path),
        }

        append_jsonl(standard_output_path, standard_record)

    print(f"Parsed Gemini batch output to: {standard_output_path}")
    print(f"Success-ish text records: {n_success:,}")
    print(f"Errors/empty:             {n_error:,}")

    return standard_output_path

In [195]:
gemini_main_plan_csv_path = Path(
    "ai_data/gemini_batches/gemini__gemini-2.5-pro__neutral_main_t1__20260428_230353__batch_plan.csv"
)

gemini_main_standard_path = parse_gemini_downloaded_batch_jsonl_to_standard_jsonl(
    downloaded_output_path=gemini_main_output_path,
    plan_csv_path=gemini_main_plan_csv_path,
    scenario_name="neutral_main_t1",
    provider_name="gemini",
    model_name="gemini-2.5-pro",
)

gemini_main_df = pd.DataFrame(read_jsonl(gemini_main_standard_path))

print(gemini_main_df.shape)
print(gemini_main_df["status"].value_counts(dropna=False))
gemini_main_df[["prompt_id", "temperature", "run_idx", "status", "text"]].head()

Parsed Gemini batch output to: ai_data/story_generations/gemini__gemini-2.5-pro__neutral_main_t1.jsonl
Success-ish text records: 28
Errors/empty:             122
(150, 25)
status
empty_text    122
success        28
Name: count, dtype: int64


,prompt_id,temperature,run_idx,status,text
0,10491,1.0,0,empty_text,None
1,10491,1.0,1,empty_text,None
2,10491,1.0,2,empty_text,None
3,10491,1.0,3,empty_text,None
4,10491,1.0,4,empty_text,None


In [196]:
PROVIDER_NAME = "gemini"
MODEL_NAME = "gemini-2.5-flash"

In [198]:
# Gemini 2.5 Flash supports the original temperature grid.
# T=1.0 neutral is covered by neutral_main_t1.

SCENARIOS["gemini_neutral_temperature_robustness"] = GenerationScenario(
    scenario_name="gemini_neutral_temperature_robustness",
    condition_type="neutral",
    temperatures=[0.7, 1.3],
    n_per_prompt=10,
    prompt_ids=SELECTED_PROMPT_IDS,
    include_personas=False,
    max_output_tokens=800,
    description="Gemini temperature robustness: neutral prompt at T=0.7 and T=1.3.",
)

SCENARIOS["gemini_personality_grid"] = GenerationScenario(
    scenario_name="gemini_personality_grid",
    condition_type="personality",
    temperatures=[0.7, 1.0, 1.3],
    n_per_prompt=10,
    prompt_ids=SELECTED_PROMPT_IDS,
    include_personas=True,
    max_output_tokens=800,
    description="Gemini personality robustness: 32 Big-Five-style profiles crossed with temperature.",
)

In [199]:
test_response = gemini_client.models.generate_content(
    model=MODEL_NAME,
    contents=[
        types.Content(
            role="user",
            parts=[types.Part(text=build_user_prompt(selected_story_prompts.loc[0, "prompt"]))]
        )
    ],
    config=types.GenerateContentConfig(
        system_instruction=NEUTRAL_SYSTEM_INSTRUCTIONS,
        temperature=1.3,
        max_output_tokens=800,
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)

print(test_response.text)

The old house stood silent, a sentinel of forgotten sorrow. I’d heard the stories, of course, the whispers of the family who vanished. Tonight, I was to debunk them. 

Inside, a single rocking chair swayed in the inky blackness. No breeze, no open window. I laughed, a nervous, forced sound. As I turned to leave, a child's giggle echoed from the floorboards. 

Then, a tiny hand wrapped around my ankle. Cold. So cold. 

"Don't go," a voice like rusted hinges scraped, "we've been waiting for you."


In [200]:
def make_gemini_batch_jsonl_file(
    scenario_name: str,
    model_name: str,
    prompts_df: pd.DataFrame,
    output_dir: Path = GEMINI_BATCH_DIR,
    exclude_existing_successes: bool = True,
    gemini_max_output_tokens: int = 800,
    thinking_budget: int = 0,
) -> tuple[Path, Path, pd.DataFrame]:
    """
    Create a Gemini batch JSONL file.

    For Gemini 2.5 Flash:
    - thinkingBudget=0 disables hidden thinking.
    - maxOutputTokens=800 is enough for these short creative-writing prompts.
    """
    standard_output_path = AI_DATA_DIR / f"gemini__{model_name}__{scenario_name}.jsonl"

    plan_df = build_gemini_generation_plan(
        scenario_name=scenario_name,
        model_name=model_name,
        prompts_df=prompts_df,
    )

    if exclude_existing_successes and standard_output_path.exists():
        done_keys = successful_request_keys(standard_output_path)
        plan_df = plan_df[~plan_df["request_key"].isin(done_keys)].copy()

    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"gemini__{model_name}__{scenario_name}__{timestamp}"

    plan_csv_path = output_dir / f"{stem}__batch_plan.csv"
    batch_jsonl_path = output_dir / f"{stem}__batch_input.jsonl"

    plan_df.to_csv(plan_csv_path, index=False)

    with open(batch_jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            request_obj = {
                "key": row["gemini_custom_id"],
                "request": {
                    "contents": [
                        {
                            "role": "user",
                            "parts": [
                                {"text": row["user_prompt"]}
                            ],
                        }
                    ],
                    "systemInstruction": {
                        "parts": [
                            {"text": row["system_instructions"]}
                        ]
                    },
                    "generationConfig": {
                        "temperature": float(row["temperature"]),
                        "maxOutputTokens": int(gemini_max_output_tokens),
                        "responseModalities": ["TEXT"],
                        "thinkingConfig": {
                            "thinkingBudget": int(thinking_budget)
                        },
                    },
                },
            }

            f.write(json.dumps(request_obj, ensure_ascii=False) + "\n")

    print(f"Scenario: {scenario_name}")
    print(f"Requests written: {len(plan_df):,}")
    print(f"Model: {model_name}")
    print(f"Gemini maxOutputTokens: {gemini_max_output_tokens}")
    print(f"Gemini thinkingBudget:  {thinking_budget}")
    print(f"Plan CSV:     {plan_csv_path}")
    print(f"Batch JSONL:  {batch_jsonl_path}")

    return batch_jsonl_path, plan_csv_path, plan_df

In [201]:
gemini_flash_main_batch = submit_gemini_batch_from_jsonl(
    scenario_name="neutral_main_t1",
    model_name=MODEL_NAME,
    prompts_df=selected_story_prompts,
    exclude_existing_successes=True,
)

gemini_flash_main_batch

Scenario: neutral_main_t1
Requests written: 150
Model: gemini-2.5-flash
Gemini maxOutputTokens: 800
Gemini thinkingBudget:  0
Plan CSV:     ai_data/gemini_batches/gemini__gemini-2.5-flash__neutral_main_t1__20260428_232058__batch_plan.csv
Batch JSONL:  ai_data/gemini_batches/gemini__gemini-2.5-flash__neutral_main_t1__20260428_232058__batch_input.jsonl
Uploaded file: files/gce21xrbris8
Submitted Gemini batch:
{
  "batch_name": "batches/3t7op2fzq1bhnkmb7k4i0rt3oiv3h260hwj7",
  "scenario_name": "neutral_main_t1",
  "model": "gemini-2.5-flash",
  "submitted_at_utc": "2026-04-29T03:21:00.516905+00:00",
  "n_requests_submitted": 150,
  "plan_csv_path": "ai_data/gemini_batches/gemini__gemini-2.5-flash__neutral_main_t1__20260428_232058__batch_plan.csv",
  "batch_jsonl_path": "ai_data/gemini_batches/gemini__gemini-2.5-flash__neutral_main_t1__20260428_232058__batch_input.jsonl",
  "uploaded_file_name": "files/gce21xrbris8",
  "raw_state": "JobState.JOB_STATE_PENDING"
}


{'batch_name': 'batches/3t7op2fzq1bhnkmb7k4i0rt3oiv3h260hwj7',
 'scenario_name': 'neutral_main_t1',
 'model': 'gemini-2.5-flash',
 'submitted_at_utc': '2026-04-29T03:21:00.516905+00:00',
 'n_requests_submitted': 150,
 'plan_csv_path': 'ai_data/gemini_batches/gemini__gemini-2.5-flash__neutral_main_t1__20260428_232058__batch_plan.csv',
 'batch_jsonl_path': 'ai_data/gemini_batches/gemini__gemini-2.5-flash__neutral_main_t1__20260428_232058__batch_input.jsonl',
 'uploaded_file_name': 'files/gce21xrbris8',
 'raw_state': 'JobState.JOB_STATE_PENDING'}

In [221]:
gemini_flash_main_status = check_gemini_batch(
    gemini_flash_main_batch["batch_name"]
)

name='batches/3t7op2fzq1bhnkmb7k4i0rt3oiv3h260hwj7' display_name='gemini__gemini-2.5-flash__neutral_main_t1__20260428_232058__batch_input' state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'> error=None create_time=datetime.datetime(2026, 4, 29, 3, 20, 59, 948638, tzinfo=TzInfo(UTC)) start_time=None end_time=datetime.datetime(2026, 4, 29, 4, 41, 10, 94272, tzinfo=TzInfo(UTC)) update_time=datetime.datetime(2026, 4, 29, 4, 41, 10, 94272, tzinfo=TzInfo(UTC)) model='models/gemini-2.5-flash' src=None dest=BatchJobDestination(
  file_name='files/batch-3t7op2fzq1bhnkmb7k4i0rt3oiv3h260hwj7'
)


In [222]:
gemini_flash_main_output_path = download_gemini_batch_output_file(
    gemini_flash_main_batch["batch_name"],
    output_dir=GEMINI_BATCH_DIR,
)

gemini_flash_main_standard_path = parse_gemini_downloaded_batch_jsonl_to_standard_jsonl(
    downloaded_output_path=gemini_flash_main_output_path,
    plan_csv_path=gemini_flash_main_batch["plan_csv_path"],
    scenario_name="neutral_main_t1",
    provider_name="gemini",
    model_name=MODEL_NAME,
)

gemini_flash_main_df = pd.DataFrame(read_jsonl(gemini_flash_main_standard_path))

print(gemini_flash_main_df.shape)
print(gemini_flash_main_df["status"].value_counts(dropna=False))
gemini_flash_main_df[["prompt_id", "temperature", "run_idx", "status", "text"]].head()

Batch state: JobState.JOB_STATE_SUCCEEDED
Batch error: None
Batch dest: format=None gcs_uri=None bigquery_uri=None file_name='files/batch-3t7op2fzq1bhnkmb7k4i0rt3oiv3h260hwj7' inlined_responses=None inlined_embed_content_responses=None
Downloaded Gemini batch output to: ai_data/gemini_batches/batches_3t7op2fzq1bhnkmb7k4i0rt3oiv3h260hwj7__output.jsonl
File size: 143,511 bytes
Parsed Gemini batch output to: ai_data/story_generations/gemini__gemini-2.5-flash__neutral_main_t1.jsonl
Success-ish text records: 150
Errors/empty:             0
(150, 25)
status
success    150
Name: count, dtype: int64


,prompt_id,temperature,run_idx,status,text
0,93855,1.0,0,success,"[ FF ] A century of smiles, tears, growth, lov..."
1,93855,1.0,1,success,Loved. Lost. Lived. Learned. Laughed. Cried. D...
2,93855,1.0,2,success,"A century: love, loss, quiet joys, changing wo..."
3,93855,1.0,3,success,"[ FF ] She lived, loved, lost, learned, laughe..."
4,93855,1.0,4,success,"Birth, love, loss, joy, sorrow, peace. Then, n..."


In [223]:
gemini_flash_main_df["word_count"] = (
    gemini_flash_main_df["text"]
    .fillna("")
    .str.findall(r"\b[\w']+\b")
    .str.len()
)

gemini_flash_main_df.groupby("prompt_id").agg(
    n=("text", "size"),
    mean_words=("word_count", "mean"),
    median_words=("word_count", "median"),
    min_words=("word_count", "min"),
    max_words=("word_count", "max"),
)

,n,mean_words,median_words,min_words,max_words
prompt_id,,,,,
10491,50,93.96,94.5,62,117
93742,50,92.74,93.0,78,108
93855,50,105.44,108.0,75,134


In [224]:
gemini_flash_temp_batch = submit_gemini_batch_from_jsonl(
    scenario_name="gemini_neutral_temperature_robustness",
    model_name=MODEL_NAME,
    prompts_df=selected_story_prompts,
    exclude_existing_successes=True,
)

gemini_flash_personality_batch = submit_gemini_batch_from_jsonl(
    scenario_name="gemini_personality_grid",
    model_name=MODEL_NAME,
    prompts_df=selected_story_prompts,
    exclude_existing_successes=True,
)

gemini_flash_batches = pd.DataFrame([
    gemini_flash_temp_batch,
    gemini_flash_personality_batch,
])

manifest_path = GEMINI_BATCH_DIR / f"gemini__{MODEL_NAME}__robustness_batch_manifest.csv"
gemini_flash_batches.to_csv(manifest_path, index=False)

gemini_flash_batches

Scenario: gemini_neutral_temperature_robustness
Requests written: 60
Model: gemini-2.5-flash
Gemini maxOutputTokens: 800
Gemini thinkingBudget:  0
Plan CSV:     ai_data/gemini_batches/gemini__gemini-2.5-flash__gemini_neutral_temperature_robustness__20260429_004143__batch_plan.csv
Batch JSONL:  ai_data/gemini_batches/gemini__gemini-2.5-flash__gemini_neutral_temperature_robustness__20260429_004143__batch_input.jsonl
Uploaded file: files/sh5ctqw5kh7q
Submitted Gemini batch:
{
  "batch_name": "batches/04ue488n3gstkeygho2c45s1pxnfzjn2f0mc",
  "scenario_name": "gemini_neutral_temperature_robustness",
  "model": "gemini-2.5-flash",
  "submitted_at_utc": "2026-04-29T04:41:45.960086+00:00",
  "n_requests_submitted": 60,
  "plan_csv_path": "ai_data/gemini_batches/gemini__gemini-2.5-flash__gemini_neutral_temperature_robustness__20260429_004143__batch_plan.csv",
  "batch_jsonl_path": "ai_data/gemini_batches/gemini__gemini-2.5-flash__gemini_neutral_temperature_robustness__20260429_004143__batch_inp

,batch_name,scenario_name,model,submitted_at_utc,n_requests_submitted,plan_csv_path,batch_jsonl_path,uploaded_file_name,raw_state
0,batches/04ue488n3gstkeygho2c45s1pxnfzjn2f0mc,gemini_neutral_temperature_robustness,gemini-2.5-flash,2026-04-29T04:41:45.960086+00:00,60,ai_data/gemini_batches/gemini__gemini-2.5-flas...,ai_data/gemini_batches/gemini__gemini-2.5-flas...,files/sh5ctqw5kh7q,JobState.JOB_STATE_PENDING
1,batches/m175ydfgalimggey0ln3mdpmepwgmimrw7s8,gemini_personality_grid,gemini-2.5-flash,2026-04-29T04:41:49.029307+00:00,2880,ai_data/gemini_batches/gemini__gemini-2.5-flas...,ai_data/gemini_batches/gemini__gemini-2.5-flas...,files/5re7pksak07a,JobState.JOB_STATE_PENDING


In [244]:
for _, row in gemini_flash_batches.iterrows():
    print("=" * 100)
    print(row["scenario_name"])
    check_gemini_batch(row["batch_name"])

gemini_neutral_temperature_robustness
name='batches/04ue488n3gstkeygho2c45s1pxnfzjn2f0mc' display_name='gemini__gemini-2.5-flash__gemini_neutral_temperature_robustness__20260429_004143__batch_input' state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'> error=None create_time=datetime.datetime(2026, 4, 29, 4, 41, 45, 221651, tzinfo=TzInfo(UTC)) start_time=None end_time=datetime.datetime(2026, 4, 29, 6, 37, 45, 401737, tzinfo=TzInfo(UTC)) update_time=datetime.datetime(2026, 4, 29, 6, 37, 45, 401737, tzinfo=TzInfo(UTC)) model='models/gemini-2.5-flash' src=None dest=BatchJobDestination(
  file_name='files/batch-04ue488n3gstkeygho2c45s1pxnfzjn2f0mc'
)
gemini_personality_grid
name='batches/m175ydfgalimggey0ln3mdpmepwgmimrw7s8' display_name='gemini__gemini-2.5-flash__gemini_personality_grid__20260429_004146__batch_input' state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'> error=None create_time=datetime.datetime(2026, 4, 29, 4, 41, 47, 899861, tzinfo=TzInfo(UTC)) start_time=None

In [245]:
parsed_paths = []

for _, row in gemini_flash_batches.iterrows():
    print("=" * 100)
    print(row["scenario_name"])

    output_path = download_gemini_batch_output_file(
        row["batch_name"],
        output_dir=GEMINI_BATCH_DIR,
    )

    standard_path = parse_gemini_downloaded_batch_jsonl_to_standard_jsonl(
        downloaded_output_path=output_path,
        plan_csv_path=row["plan_csv_path"],
        scenario_name=row["scenario_name"],
        provider_name="gemini",
        model_name=MODEL_NAME,
    )

    parsed_paths.append(standard_path)

parsed_paths

gemini_neutral_temperature_robustness
Batch state: JobState.JOB_STATE_SUCCEEDED
Batch error: None
Batch dest: format=None gcs_uri=None bigquery_uri=None file_name='files/batch-04ue488n3gstkeygho2c45s1pxnfzjn2f0mc' inlined_responses=None inlined_embed_content_responses=None
Downloaded Gemini batch output to: ai_data/gemini_batches/batches_04ue488n3gstkeygho2c45s1pxnfzjn2f0mc__output.jsonl
File size: 57,772 bytes
Parsed Gemini batch output to: ai_data/story_generations/gemini__gemini-2.5-flash__gemini_neutral_temperature_robustness.jsonl
Success-ish text records: 60
Errors/empty:             0
gemini_personality_grid
Batch state: JobState.JOB_STATE_SUCCEEDED
Batch error: None
Batch dest: format=None gcs_uri=None bigquery_uri=None file_name='files/batch-m175ydfgalimggey0ln3mdpmepwgmimrw7s8' inlined_responses=None inlined_embed_content_responses=None
Downloaded Gemini batch output to: ai_data/gemini_batches/batches_m175ydfgalimggey0ln3mdpmepwgmimrw7s8__output.jsonl
File size: 2,800,029 byt

[PosixPath('ai_data/story_generations/gemini__gemini-2.5-flash__gemini_neutral_temperature_robustness.jsonl'),
 PosixPath('ai_data/story_generations/gemini__gemini-2.5-flash__gemini_personality_grid.jsonl')]

In [246]:
all_gemini_flash_story_df = load_all_generation_files(
    provider_name="gemini",
    model_name=MODEL_NAME,
    output_dir=AI_DATA_DIR,
)

all_gemini_flash_story_df_dedup = deduplicate_generation_records(
    all_gemini_flash_story_df
)

all_gemini_flash_story_df_dedup["analysis_scenario_name"] = (
    all_gemini_flash_story_df_dedup["scenario_name"]
)

all_gemini_flash_story_df_dedup.loc[
    all_gemini_flash_story_df_dedup["scenario_name"].isin([
        "gemini_neutral_temperature_robustness",
        "neutral_temperature_robustness",
    ]),
    "analysis_scenario_name"
] = "neutral_temperature_robustness"

all_gemini_flash_story_df_dedup.loc[
    all_gemini_flash_story_df_dedup["scenario_name"].isin([
        "gemini_personality_grid",
        "personality_grid",
    ]),
    "analysis_scenario_name"
] = "personality_grid"

print("Before dedup:", all_gemini_flash_story_df.shape)
print("After dedup: ", all_gemini_flash_story_df_dedup.shape)
print(all_gemini_flash_story_df_dedup["analysis_scenario_name"].value_counts(dropna=False))
print(all_gemini_flash_story_df_dedup["status"].value_counts(dropna=False))

Before dedup: (3090, 26)
After dedup:  (3090, 27)
analysis_scenario_name
personality_grid                  2880
neutral_main_t1                    150
neutral_temperature_robustness      60
Name: count, dtype: int64
status
success    3090
Name: count, dtype: int64


In [247]:
gemini_flash_success_df = (
    all_gemini_flash_story_df_dedup
    .query("status == 'success'")
    .copy()
)

gemini_flash_success_df.groupby(
    ["analysis_scenario_name", "temperature"], dropna=False
).size()

analysis_scenario_name          temperature
neutral_main_t1                 1.0            150
neutral_temperature_robustness  0.7             30
                                1.3             30
personality_grid                0.7            960
                                1.0            960
                                1.3            960
dtype: int64

In [248]:
personality_coverage = (
    gemini_flash_success_df
    .query("analysis_scenario_name == 'personality_grid'")
    .groupby(["prompt_id", "temperature", "persona_id"], dropna=False)
    .agg(n_success=("request_key", "size"))
    .reset_index()
)

incomplete_personality_cells = personality_coverage.query("n_success < 10")

print(f"Incomplete personality cells: {len(incomplete_personality_cells)}")
incomplete_personality_cells.head(30)

Incomplete personality cells: 0


,prompt_id,temperature,persona_id,n_success


In [249]:
gemini_flash_success_df["word_count"] = (
    gemini_flash_success_df["text"]
    .fillna("")
    .str.findall(r"\b[\w']+\b")
    .str.len()
)

length_summary = (
    gemini_flash_success_df
    .groupby(["analysis_scenario_name", "prompt_id", "temperature"], dropna=False)
    .agg(
        n=("text", "size"),
        mean_words=("word_count", "mean"),
        median_words=("word_count", "median"),
        min_words=("word_count", "min"),
        max_words=("word_count", "max"),
    )
    .reset_index()
    .sort_values(["analysis_scenario_name", "prompt_id", "temperature"])
)

length_summary

,analysis_scenario_name,prompt_id,temperature,n,mean_words,median_words,min_words,max_words
0,neutral_main_t1,10491,1.0,50,93.960000,94.5,62,117
1,neutral_main_t1,93742,1.0,50,92.740000,93.0,78,108
2,neutral_main_t1,93855,1.0,50,105.440000,108.0,75,134
3,neutral_temperature_robustness,10491,0.7,10,100.500000,99.5,88,115
4,neutral_temperature_robustness,10491,1.3,10,94.900000,94.0,82,106
5,neutral_temperature_robustness,93742,0.7,10,95.400000,96.5,84,104
6,neutral_temperature_robustness,93742,1.3,10,90.500000,91.5,81,98
7,neutral_temperature_robustness,93855,0.7,10,102.300000,105.0,68,118
8,neutral_temperature_robustness,93855,1.3,10,106.100000,107.5,91,120
9,personality_grid,10491,0.7,320,95.487500,95.0,70,138


In [250]:
success_path_pkl = CLEAN_AI_DIR / f"gemini__{MODEL_NAME}__story_generations_success_only.pkl"
success_path_csv = CLEAN_AI_DIR / f"gemini__{MODEL_NAME}__story_generations_success_only.csv"

audit_path_pkl = CLEAN_AI_DIR / f"gemini__{MODEL_NAME}__story_generations_all_records_with_errors.pkl"
audit_path_csv = CLEAN_AI_DIR / f"gemini__{MODEL_NAME}__story_generations_all_records_with_errors.csv"

gemini_flash_success_df.to_pickle(success_path_pkl)
gemini_flash_success_df.to_csv(success_path_csv, index=False)

all_gemini_flash_story_df_dedup.to_pickle(audit_path_pkl)
all_gemini_flash_story_df_dedup.to_csv(audit_path_csv, index=False)

print(success_path_pkl)
print(success_path_csv)
print(audit_path_pkl)
print(audit_path_csv)

ai_data/processed/gemini__gemini-2.5-flash__story_generations_success_only.pkl
ai_data/processed/gemini__gemini-2.5-flash__story_generations_success_only.csv
ai_data/processed/gemini__gemini-2.5-flash__story_generations_all_records_with_errors.pkl
ai_data/processed/gemini__gemini-2.5-flash__story_generations_all_records_with_errors.csv
